# EventInsight — Trabalho Prático 2

## 0. Introdução

O EventInsight é uma proposta de sistema de apoio à decisão para produtoras de eventos musicais. A solução busca estimar público, apoiar simulações de lucro e analisar alternativas de preço de ingresso.

O Trabalho 2 continua o projeto desenvolvido no Trabalho 1. O objetivo não é apenas aumentar a complexidade do código, mas revisar criticamente os dados e os modelos, justificar as decisões, comparar estratégias e avaliar a possibilidade de implantação em um contexto real.

Nesta versão serão desenvolvidas as seguintes etapas:

1. revisão e preparação da base;
2. análise exploratória orientada à modelagem;
3. diagnóstico do modelo inicial;
4. engenharia e seleção de atributos;
5. comparação e ajuste de modelos;
6. validação cruzada e análise de erros;
7. simulações de negócio;
8. viabilidade técnica e econômica.

## 1. Configuração do projeto

Esta seção centraliza as bibliotecas, a semente aleatória e os caminhos utilizados. A semente torna os experimentos reproduzíveis: execuções com a mesma configuração produzem as mesmas divisões e resultados aleatórios.

O backend gráfico `Agg` permite gerar figuras tanto no Jupyter/Colab quanto em ambientes sem interface gráfica.

In [ ]:
from pathlib import Path
import os
import re

SEED = 42
PASTA_PROJETO = Path.cwd()
PASTA_DADOS = PASTA_PROJETO / "inputs"
PASTA_EDA = PASTA_PROJETO / "outputs" / "eda"
PASTA_EDA.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(PASTA_PROJETO / ".cache_matplotlib"))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

print(f"Pasta do projeto: {PASTA_PROJETO}")
print(f"Seed: {SEED}")

## 2. Carregamento dos dados

### 2.1 Leitura do arquivo

A base é lida uma única vez e o arquivo original não é alterado. O caminho relativo evita dependência de uma pasta pessoal. No Google Colab, deve-se criar a pasta `inputs` e enviar `baseinicial.csv` para ela.

In [ ]:
CAMINHO_DADOS = PASTA_DADOS / "baseinicial.csv"

if not CAMINHO_DADOS.exists():
    raise FileNotFoundError(
        f"Base não encontrada em {CAMINHO_DADOS}. "
        "Coloque baseinicial.csv dentro da pasta inputs."
    )

dados_brutos = pd.read_csv(CAMINHO_DADOS)
print(f"Base carregada: {dados_brutos.shape[0]} linhas e {dados_brutos.shape[1]} colunas.")

### 2.2 Inspeção inicial

Antes de renomear ou converter colunas, inspecionamos o conteúdo recebido. Essa verificação revelou que o cabeçalho do CSV está deslocado em relação aos valores: `Festival_Name` contém países, `Location` contém público e `Attendance_Numbers` contém a faixa etária.

In [ ]:
display(dados_brutos.head())
print("Colunas originais:")
print(dados_brutos.columns.tolist())

## 3. Limpeza e pré-processamento

### 3.1 Validação da estrutura

As colunas obrigatórias são validadas antes da limpeza. Isso impede que uma alteração no arquivo produza um modelo treinado silenciosamente com variáveis trocadas.

In [ ]:
colunas_obrigatorias = {
    "Festival_Name", "Location", "Attendance_Numbers",
    "Visitor_Demographics", "Economic_Impact", "Music_Genre",
}
colunas_ausentes = colunas_obrigatorias.difference(dados_brutos.columns)

if colunas_ausentes:
    raise ValueError(f"Colunas esperadas não encontradas: {sorted(colunas_ausentes)}")

print("Estrutura mínima validada.")

### 3.2 Correção semântica das colunas

No Trabalho 1, os nomes eram substituídos apenas pela posição. Agora a correspondência é declarada explicitamente a partir do conteúdo observado. Essa estratégia melhora rastreabilidade e reduz o risco de erros silenciosos.

In [ ]:
df_limpo = pd.DataFrame({
    "pais": dados_brutos["Festival_Name"].astype("string").str.strip(),
    "audience": pd.to_numeric(dados_brutos["Location"], errors="coerce"),
    "faixa_etaria": dados_brutos["Attendance_Numbers"].astype("string").str.strip(),
    "tipo_publico": dados_brutos["Visitor_Demographics"].astype("string").str.strip(),
    "impacto_original": dados_brutos["Economic_Impact"].astype("string").str.strip(),
    "genero": dados_brutos["Music_Genre"].astype("string").str.strip(),
})

display(df_limpo.head())

### 3.3 Tratamento do impacto econômico

O Trabalho 1 removia os símbolos monetários e mantinha somente o número. Isso fazia valores de moedas diferentes parecerem comparáveis.

Nesta versão preservamos:

- o texto original;
- o valor nominal em milhões;
- o prefixo monetário.

O valor nominal permite reproduzir o experimento anterior, mas ainda não representa uma moeda comum. Alguns símbolos também estão corrompidos no arquivo. Portanto, o impacto será tratado como atributo exploratório até que sua disponibilidade e unidade sejam validadas.

In [ ]:
def extrair_impacto_milhoes(valor):
    encontrado = re.search(r"([\d.,]+)", str(valor))
    if encontrado is None:
        return pd.NA
    return float(encontrado.group(1).replace(",", "."))


def extrair_moeda(valor):
    prefixo = re.match(r"\s*([^\d]+)", str(valor))
    if prefixo is None:
        return "nao_identificada"
    moeda = prefixo.group(1).strip()
    return "simbolo_corrompido" if any(ord(c) > 127 for c in moeda) else moeda


df_limpo["impacto_milhoes_nominais"] = df_limpo["impacto_original"].apply(extrair_impacto_milhoes)
df_limpo["moeda"] = df_limpo["impacto_original"].apply(extrair_moeda)

display(df_limpo[["impacto_original", "impacto_milhoes_nominais", "moeda"]].head(10))

### 3.4 Conversão da faixa etária

A faixa etária é separada inicialmente em idade mínima e máxima. Nesta etapa fazemos apenas a conversão estrutural; idade central e amplitude serão criadas posteriormente como engenharia de atributos.

In [ ]:
idades = df_limpo["faixa_etaria"].str.extract(
    r"(?P<idade_min>\d+)\D+(?P<idade_max>\d+)"
)
df_limpo[["idade_min", "idade_max"]] = idades.astype("float64")

display(df_limpo[["faixa_etaria", "idade_min", "idade_max"]].head())

### 3.5 Valores ausentes e tipos

Registros sem `audience` não podem ser utilizados em regressão supervisionada. Registros sem impacto extraível também impediriam a reprodução do modelo do Trabalho 1. A quantidade removida é registrada para que a limpeza seja auditável.

In [ ]:
antes_nulos = len(df_limpo)
df_limpo = df_limpo.dropna(
    subset=["audience", "impacto_milhoes_nominais", "idade_min", "idade_max"]
).copy()
removidos_nulos = antes_nulos - len(df_limpo)

df_limpo["audience"] = df_limpo["audience"].astype("int64")
df_limpo["impacto_milhoes_nominais"] = df_limpo["impacto_milhoes_nominais"].astype("float64")

print(f"Registros removidos por campos essenciais ausentes: {removidos_nulos}")

### 3.6 Duplicatas

Uma duplicata somente deve ser removida quando todas as variáveis disponíveis coincidirem. Duas linhas de Fiji possuem exatamente o mesmo país, público, faixa etária, perfil, impacto e gênero. Manter ambas atribuiria peso duplo artificial ao mesmo padrão.

In [ ]:
colunas_identificacao = [
    "pais", "audience", "faixa_etaria", "tipo_publico",
    "impacto_original", "genero",
]

mascara_duplicadas = df_limpo.duplicated(
    subset=colunas_identificacao, keep=False
)
display(df_limpo.loc[mascara_duplicadas, colunas_identificacao])

quantidade_duplicadas = int(
    df_limpo.duplicated(subset=colunas_identificacao).sum()
)
df_limpo = df_limpo.drop_duplicates(
    subset=colunas_identificacao, keep="first"
).reset_index(drop=True)

print(f"Duplicatas exatas removidas: {quantidade_duplicadas}")
print(f"Dimensão final após limpeza: {df_limpo.shape}")

### 3.7 Verificação final da limpeza

`info()` confirma os tipos e valores não nulos. `describe()` verifica escala, dispersão, mínimos e máximos antes que possíveis erros de conversão sejam confundidos com características dos dados.

In [ ]:
df_limpo.info()
display(df_limpo.describe().round(2))

## 4. Análise Exploratória dos Dados (EDA)

A EDA desta versão não serve apenas para ilustrar a base. Ela deve orientar a seleção de atributos, a escolha das métricas, o diagnóstico de overfitting e a avaliação da aplicabilidade real.

### 4.1 Distribuição do público

Comparamos a escala original e `log(1 + audience)`. A transformação logarítmica reduz a influência visual dos maiores eventos e será posteriormente testada como transformação do alvo. Ela não será adotada sem comparação por validação cruzada na escala real.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_limpo["audience"], bins=20, kde=True, ax=eixos[0])
eixos[0].set(title="Distribuição do público", xlabel="Público", ylabel="Eventos")
sns.histplot(np.log1p(df_limpo["audience"]), bins=20, kde=True, ax=eixos[1])
eixos[1].set(title="Distribuição de log(1 + público)", xlabel="log(1 + público)", ylabel="Eventos")
fig.tight_layout()
fig.savefig(PASTA_EDA / "01_distribuicao_publico.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"Assimetria na escala original: {df_limpo['audience'].skew():.3f}")
print(f"Assimetria após log1p: {np.log1p(df_limpo['audience']).skew():.3f}")

### 4.2 Boxplot do público

O boxplot identifica eventos distantes da maior parte da amostra. Esses casos não serão removidos automaticamente: um evento extremo pode ser válido e relevante para o negócio.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(x=df_limpo["audience"], ax=ax)
ax.set(title="Dispersão e eventos de maior público", xlabel="Público")
fig.tight_layout()
fig.savefig(PASTA_EDA / "02_boxplot_publico.png", dpi=180, bbox_inches="tight")
plt.show()

### 4.3 Impacto nominal versus público

Este gráfico reproduz e aprofunda a principal relação usada no Trabalho 1. As escalas logarítmicas facilitam a visualização, mas não corrigem a mistura de moedas nem garantem que o impacto esteja disponível antes do evento.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df_limpo,
    x="impacto_milhoes_nominais",
    y="audience",
    alpha=0.75,
    ax=ax,
)
ax.set(
    xscale="log", yscale="log",
    title="Impacto nominal e público (escalas logarítmicas)",
    xlabel="Impacto em milhões na moeda original",
    ylabel="Público",
)
fig.tight_layout()
fig.savefig(PASTA_EDA / "03_impacto_vs_publico.png", dpi=180, bbox_inches="tight")
plt.show()

### 4.4 Correlações de Pearson e Spearman

Pearson mede associação linear; Spearman mede associação monotônica e é menos sensível à escala. Correlação será apenas uma das técnicas de seleção, pois relações não lineares podem ser relevantes e correlação não demonstra causalidade.

In [ ]:
colunas_numericas = ["audience", "impacto_milhoes_nominais", "idade_min", "idade_max"]
corr_pearson = df_limpo[colunas_numericas].corr(method="pearson")
corr_spearman = df_limpo[colunas_numericas].corr(method="spearman")

fig, eixos = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(corr_pearson, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, ax=eixos[0])
eixos[0].set_title("Correlação de Pearson")
sns.heatmap(corr_spearman, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, ax=eixos[1])
eixos[1].set_title("Correlação de Spearman")
fig.tight_layout()
fig.savefig(PASTA_EDA / "04_correlacoes.png", dpi=180, bbox_inches="tight")
plt.show()

print("Impacto x público — Pearson:", round(corr_pearson.loc["impacto_milhoes_nominais", "audience"], 3))
print("Impacto x público — Spearman:", round(corr_spearman.loc["impacto_milhoes_nominais", "audience"], 3))

### 4.5 Frequência de gêneros e países

A análise utiliza frequência em vez de público médio. Como muitas categorias possuem somente uma observação, médias por categoria poderiam criar rankings instáveis e enganosos. A alta cardinalidade será tratada posteriormente na engenharia e seleção de atributos.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(16, 7))

df_limpo["genero"].value_counts().head(15).sort_values().plot(kind="barh", ax=eixos[0])
eixos[0].set(title="15 gêneros mais frequentes", xlabel="Quantidade de eventos", ylabel="")

df_limpo["pais"].value_counts().head(15).sort_values().plot(kind="barh", ax=eixos[1])
eixos[1].set(title="15 países mais frequentes", xlabel="Quantidade de eventos", ylabel="")

fig.tight_layout()
fig.savefig(PASTA_EDA / "05_frequencias_categoricas.png", dpi=180, bbox_inches="tight")
plt.show()

prop_paises_unicos = (df_limpo["pais"].value_counts() == 1).mean()
prop_generos_unicos = (df_limpo["genero"].value_counts() == 1).mean()
print(f"Países com uma observação: {prop_paises_unicos:.1%}")
print(f"Gêneros com uma observação: {prop_generos_unicos:.1%}")

### 4.6 Distribuição do impacto nominal

O gráfico descreve os números registrados, mas não deve ser interpretado como uma distribuição monetária global porque as moedas ainda não foram convertidas para uma unidade comum.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df_limpo["impacto_milhoes_nominais"], bins=20, kde=True, ax=ax)
ax.set(
    title="Distribuição do impacto nominal",
    xlabel="Milhões na moeda original",
    ylabel="Eventos",
)
fig.tight_layout()
fig.savefig(PASTA_EDA / "06_distribuicao_impacto_nominal.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"Assimetria do impacto nominal: {df_limpo['impacto_milhoes_nominais'].skew():.3f}")

### 4.7 Síntese da EDA

Os resultados levantam quatro hipóteses para as próximas etapas:

1. testar o alvo na escala original e logarítmica;
2. avaliar erros absolutos e relativos devido à diferença de escala entre eventos;
3. verificar se a forte relação com impacto representa informação disponível em produção;
4. reduzir a cardinalidade de país, gênero e tipo de público para diminuir dimensionalidade e risco de memorização.

In [ ]:
resumo_eda = pd.DataFrame({
    "métrica": [
        "Assimetria do público",
        "Assimetria do impacto nominal",
        "Correlação Pearson: impacto x público",
        "Correlação Spearman: impacto x público",
        "Proporção de países com uma observação",
        "Proporção de gêneros com uma observação",
    ],
    "valor": [
        df_limpo["audience"].skew(),
        df_limpo["impacto_milhoes_nominais"].skew(),
        corr_pearson.loc["impacto_milhoes_nominais", "audience"],
        corr_spearman.loc["impacto_milhoes_nominais", "audience"],
        prop_paises_unicos,
        prop_generos_unicos,
    ],
})
display(resumo_eda)
resumo_eda.to_csv(PASTA_EDA / "resumo_eda.csv", index=False, encoding="utf-8-sig")

## 5. Diagnóstico do modelo do Trabalho 1

Antes de propor novos atributos ou algoritmos, reproduzimos a estratégia central do Trabalho 1. Essa etapa estabelece uma referência comparável e permite identificar limitações concretas.

O modelo original utiliza:

- `impacto_milhoes_nominais` como único atributo;
- `log(1 + audience)` como variável-alvo;
- divisão aleatória de 80% para treino e 20% para teste;
- Random Forest com 100 árvores e `random_state=42`.

Nesta seção não haverá ajuste de hiperparâmetros. Alterar o modelo antes de medir a referência impediria uma comparação justa entre os trabalhos.

### 5.1 Preparação do experimento de referência

Mantemos a mesma semente e configuração geral do Trabalho 1. A transformação logarítmica é invertida antes do cálculo das métricas, permitindo interpretar os erros em quantidade de pessoas.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X_inicial = df_limpo[["impacto_milhoes_nominais"]]
y_real = df_limpo["audience"]
y_log = np.log1p(y_real)

X_treino, X_teste, y_treino_log, y_teste_log, y_treino_real, y_teste_real = train_test_split(
    X_inicial,
    y_log,
    y_real,
    test_size=0.20,
    random_state=SEED,
)

print(f"Eventos de treino: {len(X_treino)}")
print(f"Eventos de teste: {len(X_teste)}")

### 5.2 Métricas utilizadas

Nenhuma métrica isolada descreve adequadamente eventos com escalas muito diferentes:

- **MAE:** erro absoluto médio em pessoas; é fácil de interpretar;
- **RMSE:** penaliza mais os erros muito grandes;
- **R²:** mede a proporção da variação explicada, mas pode ser instável em amostras pequenas;
- **MAPE:** expressa erro percentual médio, porém dá maior peso relativo aos menores eventos;
- **WAPE:** divide a soma dos erros absolutos pela soma do público e fornece uma medida percentual agregada.

Como não existem públicos iguais a zero nesta base, o MAPE pode ser calculado sem divisão por zero. Ainda assim, será interpretado junto às outras métricas.

In [ ]:
def calcular_metricas(y_verdadeiro, y_previsto):
    y_verdadeiro = np.asarray(y_verdadeiro)
    y_previsto = np.asarray(y_previsto)
    return {
        "MAE": mean_absolute_error(y_verdadeiro, y_previsto),
        "RMSE": mean_squared_error(y_verdadeiro, y_previsto) ** 0.5,
        "R2": r2_score(y_verdadeiro, y_previsto),
        "MAPE_%": np.mean(np.abs((y_verdadeiro - y_previsto) / y_verdadeiro)) * 100,
        "WAPE_%": np.sum(np.abs(y_verdadeiro - y_previsto)) / np.sum(np.abs(y_verdadeiro)) * 100,
    }

### 5.3 Baseline pela mediana

Um modelo de Machine Learning somente agrega valor se superar uma referência simples. O baseline prevê, para todos os eventos de teste, a mediana do público observada no conjunto de treino.

A mediana foi escolhida porque minimiza o erro absoluto e é menos influenciada pelos megaeventos do que a média.

In [ ]:
mediana_treino = float(np.median(y_treino_real))
previsao_baseline = np.full(len(y_teste_real), mediana_treino)
metricas_baseline = calcular_metricas(y_teste_real, previsao_baseline)

display(pd.DataFrame([metricas_baseline], index=["Baseline — mediana"]).round(3))

### 5.4 Reprodução do Random Forest inicial

O Random Forest é treinado com a configuração original. As previsões em log são convertidas novamente para número de pessoas com `expm1`.

In [ ]:
modelo_inicial = RandomForestRegressor(
    n_estimators=100,
    random_state=SEED,
)
modelo_inicial.fit(X_treino, y_treino_log)

previsao_treino = np.expm1(modelo_inicial.predict(X_treino))
previsao_teste = np.expm1(modelo_inicial.predict(X_teste))

metricas_treino = calcular_metricas(y_treino_real, previsao_treino)
metricas_teste = calcular_metricas(y_teste_real, previsao_teste)

tabela_diagnostico = pd.DataFrame(
    [metricas_baseline, metricas_treino, metricas_teste],
    index=["Baseline — teste", "Random Forest — treino", "Random Forest — teste"],
)
display(tabela_diagnostico.round(3))

### 5.5 Diagnóstico preliminar de overfitting

Comparamos treino e teste. Um erro de treino muito menor que o erro de teste indica que o modelo pode estar memorizando padrões da amostra.

Essa comparação é apenas um diagnóstico inicial. Uma única divisão pode produzir resultados favoráveis ou desfavoráveis por acaso; a estabilidade será examinada formalmente na etapa de validação cruzada.

In [ ]:
razao_mae_teste_treino = metricas_teste["MAE"] / metricas_treino["MAE"]
razao_rmse_teste_treino = metricas_teste["RMSE"] / metricas_treino["RMSE"]

print(f"Razão MAE teste/treino: {razao_mae_teste_treino:.2f}")
print(f"Razão RMSE teste/treino: {razao_rmse_teste_treino:.2f}")

if razao_mae_teste_treino >= 1.5:
    print("Sinal preliminar de overfitting: o MAE de teste é pelo menos 50% maior que o de treino.")
else:
    print("A diferença de MAE não indica overfitting forte nesta divisão, mas precisa de validação cruzada.")

### 5.6 Público real versus previsto

No gráfico ideal, os pontos estariam próximos da diagonal. As escalas logarítmicas permitem visualizar eventos pequenos e grandes simultaneamente. O gráfico não substitui as métricas, mas ajuda a identificar subestimação ou superestimação sistemática.

In [ ]:
PASTA_DIAGNOSTICO = PASTA_PROJETO / "outputs" / "diagnostico_modelo_inicial"
PASTA_DIAGNOSTICO.mkdir(parents=True, exist_ok=True)

limite_min = min(y_teste_real.min(), previsao_teste.min())
limite_max = max(y_teste_real.max(), previsao_teste.max())

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_teste_real, previsao_teste, alpha=0.75)
ax.plot([limite_min, limite_max], [limite_min, limite_max], "--", color="black", label="Previsão perfeita")
ax.set(
    xscale="log", yscale="log",
    title="Modelo inicial: público real versus previsto",
    xlabel="Público real", ylabel="Público previsto",
)
ax.legend()
fig.tight_layout()
fig.savefig(PASTA_DIAGNOSTICO / "01_real_vs_previsto.png", dpi=180, bbox_inches="tight")
plt.show()

### 5.7 Resíduos do modelo inicial

O histograma verifica centralização e caudas; o segundo painel mostra se o erro muda com a escala real do evento. Este diagnóstico é preliminar porque utiliza uma única divisão treino–teste, sendo aprofundado depois com previsões externas.

In [ ]:
residuos_modelo_inicial = previsao_teste - y_teste_real
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(residuos_modelo_inicial, bins=15, kde=True, ax=eixos[0])
eixos[0].axvline(0, color="black", linewidth=1)
eixos[0].set(title="Resíduos do modelo inicial", xlabel="Previsto − real", ylabel="Eventos")
sns.scatterplot(x=y_teste_real, y=residuos_modelo_inicial, ax=eixos[1])
eixos[1].axhline(0, color="black", linewidth=1)
eixos[1].set(xscale="log", title="Resíduo versus público real", xlabel="Público real — escala log", ylabel="Previsto − real")
fig.tight_layout()
fig.savefig(PASTA_DIAGNOSTICO / "02_residuos_modelo_inicial.png", dpi=180, bbox_inches="tight")
plt.show()

### 5.8 Limitações identificadas

O resultado do modelo inicial deve ser interpretado com cautela:

1. **Apenas uma variável:** o modelo depende quase totalmente do impacto econômico;
2. **Possível vazamento:** o impacto realizado pode ser conhecido somente depois do evento;
3. **Moedas não comparáveis:** os números nominais misturam unidades monetárias;
4. **Uma única divisão:** o resultado depende dos eventos sorteados para teste;
5. **Amostra pequena:** 204 registros limitam modelos complexos;
6. **Eventos discretizados:** há apenas 26 valores distintos de público;
7. **Desbalanceamento de escala:** poucos eventos grandes influenciam RMSE e R²;
8. **Ausência de pipeline:** limpeza, transformação e modelo ainda não estão encapsulados em um único objeto reproduzível.

Essas limitações justificam as próximas etapas: engenharia e seleção de atributos, comparação de algoritmos, ajuste de hiperparâmetros, validação cruzada e análise por escala.

In [ ]:
tabela_diagnostico.to_csv(
    PASTA_DIAGNOSTICO / "metricas_modelo_inicial.csv",
    encoding="utf-8-sig",
)

diagnostico_preliminar = pd.DataFrame({
    "indicador": [
        "Razão MAE teste/treino",
        "Razão RMSE teste/treino",
        "Quantidade de atributos",
        "Valores distintos do alvo",
    ],
    "valor": [
        razao_mae_teste_treino,
        razao_rmse_teste_treino,
        X_inicial.shape[1],
        y_real.nunique(),
    ],
})
display(diagnostico_preliminar.round(3))
diagnostico_preliminar.to_csv(
    PASTA_DIAGNOSTICO / "diagnostico_preliminar.csv",
    index=False,
    encoding="utf-8-sig",
)

## 6. Engenharia de atributos

O objetivo da engenharia de atributos é representar melhor as informações disponíveis, sem criar variáveis artificiais apenas para aumentar a complexidade.

Os atributos devem atender a quatro critérios:

1. possuir interpretação relacionada ao problema;
2. estar potencialmente disponíveis no momento da previsão;
3. reduzir ruído ou dimensionalidade;
4. não utilizar direta ou indiretamente o público real.

A criação de um atributo não significa que ele entrará no modelo final. Sua utilidade será medida nas etapas de feature selection e validação cruzada.

### 6.1 Cópia da base limpa

Criamos uma cópia para preservar `df_limpo`. Assim é possível comparar os dados antes e depois da engenharia e repetir experimentos sem alterar silenciosamente a base preparada.

In [ ]:
df_atributos = df_limpo.copy()
print(f"Colunas antes da engenharia: {df_limpo.shape[1]}")

### 6.2 Idade central e amplitude etária

`idade_min` e `idade_max` são fortemente relacionadas entre si. Criamos:

- **idade_central:** ponto central da faixa informada;
- **amplitude_etaria:** largura da faixa de idade.

A idade central resume o perfil etário, enquanto a amplitude diferencia eventos direcionados a públicos específicos de eventos com alcance etário mais abrangente. As colunas originais serão preservadas para que a seleção de atributos possa comparar as representações.

In [ ]:
df_atributos["idade_central"] = (
    df_atributos["idade_min"] + df_atributos["idade_max"]
) / 2
df_atributos["amplitude_etaria"] = (
    df_atributos["idade_max"] - df_atributos["idade_min"]
)

display(
    df_atributos[
        ["faixa_etaria", "idade_min", "idade_max", "idade_central", "amplitude_etaria"]
    ].head()
)

### 6.3 Transformação logarítmica do impacto

O impacto nominal apresentou assimetria elevada. `log1p` reduz a distância entre valores pequenos e grandes e permite representar relações multiplicativas com maior estabilidade.

Esse atributo pertence somente ao **cenário de reprodução acadêmica**, pois o problema de moedas diferentes e a possível indisponibilidade do impacto antes do evento continuam existindo. A transformação logarítmica melhora a escala, mas não corrige a validade da informação.

In [ ]:
df_atributos["log_impacto_nominal"] = np.log1p(
    df_atributos["impacto_milhoes_nominais"]
)

print("Assimetria antes do log:", round(df_atributos["impacto_milhoes_nominais"].skew(), 3))
print("Assimetria após o log:", round(df_atributos["log_impacto_nominal"].skew(), 3))

### 6.4 Visualização do efeito do log

Os histogramas usam os mesmos eventos, mas escalas diferentes. O objetivo não é afirmar que o log corrige moedas ou torna o impacto disponível antes do evento; ele apenas reduz a assimetria numérica e a influência visual dos maiores valores.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df_atributos["impacto_milhoes_nominais"], bins=20, kde=True, ax=eixos[0])
eixos[0].set(title="Impacto nominal — escala original", xlabel="Milhões na moeda original", ylabel="Eventos")
sns.histplot(df_atributos["log_impacto_nominal"], bins=20, kde=True, ax=eixos[1])
eixos[1].set(title="Impacto nominal — após log1p", xlabel="log1p do impacto nominal", ylabel="Eventos")
fig.tight_layout()
plt.show()

### 6.4 Representação dos gêneros musicais

O gênero original possui 102 rótulos em apenas 204 registros. Aplicar one-hot encoding diretamente criaria muitas colunas raras e aumentaria o risco de memorização.

Além disso, categorias como `Rock/Pop` e `Pop/Rock` representam combinações semelhantes, mas seriam tratadas como categorias diferentes.

Em vez de manter apenas o primeiro gênero — o que descartaria informação — criamos indicadores binários para famílias musicais recorrentes. Um evento pode pertencer a várias famílias simultaneamente. Por exemplo, `Rock/Pop` recebe valor 1 nas colunas de rock e pop.

In [ ]:
familias_musicais = {
    "rock": ["rock"],
    "pop": ["pop"],
    "eletronica": ["electronic", "edm"],
    "jazz": ["jazz"],
    "folk": ["folk"],
    "classica": ["classical"],
    "indie": ["indie"],
    "metal": ["metal"],
    "reggae": ["reggae"],
    "alternativa": ["alternative"],
}

genero_minusculo = df_atributos["genero"].str.lower()
colunas_genero = []

for familia, termos in familias_musicais.items():
    coluna = f"genero_{familia}"
    df_atributos[coluna] = genero_minusculo.apply(
        lambda texto: int(any(termo in texto for termo in termos))
    )
    colunas_genero.append(coluna)

# Eventos sem nenhuma das famílias principais recebem o indicador "outros".
df_atributos["genero_outros"] = (
    df_atributos[colunas_genero].sum(axis=1) == 0
).astype(int)
colunas_genero.append("genero_outros")

display(df_atributos[["genero"] + colunas_genero].head(12))

### 6.5 Frequência das famílias musicais

A contagem verifica se os indicadores foram criados corretamente e revela famílias raras. Famílias com poucas observações não serão removidas automaticamente: sua permanência dependerá do desempenho fora da amostra e da interpretabilidade.

In [ ]:
frequencia_familias = (
    df_atributos[colunas_genero]
    .sum()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
frequencia_familias.plot(kind="barh", ax=ax)
ax.set(
    title="Quantidade de eventos por família musical",
    xlabel="Quantidade de eventos",
    ylabel="",
)
fig.tight_layout()

PASTA_ATRIBUTOS = PASTA_PROJETO / "outputs" / "engenharia_atributos"
PASTA_ATRIBUTOS.mkdir(parents=True, exist_ok=True)
fig.savefig(PASTA_ATRIBUTOS / "01_frequencia_familias_musicais.png", dpi=180, bbox_inches="tight")
plt.show()

display(frequencia_familias.rename("quantidade").to_frame())

### 6.6 Atributos não utilizados nesta etapa

Algumas transformações foram conscientemente adiadas ou excluídas:

- **porte e quartis:** derivam de `audience` e causariam vazamento se usados como entrada;
- **país em one-hot:** 82,6% dos países aparecem uma única vez, favorecendo memorização;
- **tipo de público em one-hot:** possui 119 descrições e forte sobreposição com gênero;
- **moeda em one-hot:** existem muitos rótulos raros e símbolos corrompidos;
- **conversão cambial:** exigiria moeda correta e data de referência, informações ausentes.

Essas variáveis podem continuar nas análises descritivas, mas não entram automaticamente no conjunto candidato do modelo.

### 6.7 Conjuntos candidatos de atributos

Serão mantidos dois cenários para responder a perguntas diferentes:

**Cenário A — reprodução acadêmica:** inclui impacto e permite comparar diretamente com o Trabalho 1.

**Cenário B — pré-evento:** exclui impacto e utiliza somente informações que poderiam estar disponíveis antes do evento. Esse cenário é mais coerente com implantação, mas a base atual possui poucos atributos pré-evento, portanto seu desempenho pode ser limitado.

Separar os cenários evita apresentar uma métrica alta como prova de viabilidade quando ela depende de uma variável potencialmente indisponível.

In [ ]:
atributos_demograficos = [
    "idade_central",
    "amplitude_etaria",
]

atributos_cenario_a = [
    "impacto_milhoes_nominais",
    "log_impacto_nominal",
] + atributos_demograficos + colunas_genero

atributos_cenario_b = atributos_demograficos + colunas_genero

print("Cenário A — reprodução acadêmica:")
print(atributos_cenario_a)
print("\nCenário B — informações pré-evento:")
print(atributos_cenario_b)

### 6.8 Auditoria contra vazamento do alvo

Antes da modelagem, verificamos explicitamente que `audience`, portes e quartis não aparecem nos conjuntos candidatos. Essa validação simples reduz o risco de incluir acidentalmente uma variável derivada do alvo.

In [ ]:
atributos_proibidos = {
    "audience", "log_audience", "porte_avaliacao", "faixa_analitica",
}

for nome_cenario, atributos in {
    "Cenário A": atributos_cenario_a,
    "Cenário B": atributos_cenario_b,
}.items():
    vazamentos = atributos_proibidos.intersection(atributos)
    assert not vazamentos, f"{nome_cenario} contém possível vazamento: {vazamentos}"
    assert not df_atributos[atributos].isna().any().any(), f"{nome_cenario} contém valores ausentes."

print("Auditoria concluída: nenhum atributo derivado do alvo foi incluído.")

### 6.9 Dicionário e exportação dos atributos

O dicionário documenta a origem e a limitação de cada variável. Essa rastreabilidade ajuda a equipe a explicar por que um atributo existe e se ele é aceitável em produção.

In [ ]:
dicionario_atributos = pd.DataFrame([
    {"atributo": "idade_central", "origem": "média entre idade mínima e máxima", "uso": "A e B", "limitação": "resume a faixa etária"},
    {"atributo": "amplitude_etaria", "origem": "idade máxima menos idade mínima", "uso": "A e B", "limitação": "não mede distribuição interna"},
    {"atributo": "impacto_milhoes_nominais", "origem": "parte numérica do impacto original", "uso": "somente A", "limitação": "moedas diferentes e possível vazamento"},
    {"atributo": "log_impacto_nominal", "origem": "log1p do impacto nominal", "uso": "somente A", "limitação": "não corrige moedas nem disponibilidade"},
    {"atributo": "genero_*", "origem": "indicadores de famílias musicais", "uso": "A e B", "limitação": "agrupamento baseado nos rótulos disponíveis"},
])

display(dicionario_atributos)
dicionario_atributos.to_csv(
    PASTA_ATRIBUTOS / "dicionario_atributos.csv",
    index=False,
    encoding="utf-8-sig",
)
df_atributos.to_csv(
    PASTA_ATRIBUTOS / "base_com_atributos.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Colunas após a engenharia: {df_atributos.shape[1]}")
print(f"Arquivos salvos em: {PASTA_ATRIBUTOS}")

## 7. Separação e diagnóstico por porte

A diferença de escala entre os eventos exige métricas segmentadas: um erro de 7 mil pessoas possui consequências distintas em um evento de 10 mil e em outro de 200 mil.

Entretanto, dividir a base sem verificar a quantidade de observações pode produzir resultados instáveis. Esta seção avalia a distribuição antes de qualquer decisão sobre modelos separados.

Os grupos são utilizados somente para análise e avaliação. Como dependem de `audience`, eles não podem ser usados como atributos nem como roteadores de previsão.

### 7.1 Faixas oficiais de avaliação

A classificação oficial do projeto passa a utilizar três faixas:

- Pequeno: até 20 mil pessoas;
- Médio: acima de 20 mil e até 80 mil;
- Grande/Mega: acima de 80 mil.

A união de Grande e Mega evita analisar os dois megaeventos isoladamente. Os limites são hipóteses do projeto, não padrões universais do setor. As faixas servirão para avaliar o modelo geral, nunca para escolher qual modelo fará a previsão.

In [ ]:
df_atributos["porte_avaliacao"] = pd.cut(
    df_atributos["audience"],
    bins=[-np.inf, 20_000, 80_000, np.inf],
    labels=["Pequeno", "Médio", "Grande/Mega"],
    include_lowest=True,
)

contagem_portes = df_atributos["porte_avaliacao"].value_counts(sort=False)
display(contagem_portes.rename("quantidade").to_frame())

### 7.2 DataFrames por porte

Os subconjuntos são criados somente para estatísticas e métricas posteriores. O treinamento continuará usando todos os eventos em um modelo geral.

In [ ]:
dados_por_porte = {
    porte: df_atributos.loc[df_atributos["porte_avaliacao"] == porte].copy()
    for porte in df_atributos["porte_avaliacao"].cat.categories
}

df_pequeno = dados_por_porte["Pequeno"]
df_medio = dados_por_porte["Médio"]
df_grande_mega = dados_por_porte["Grande/Mega"]

for nome, subconjunto in dados_por_porte.items():
    print(f"{nome}: {len(subconjunto)} eventos")

### 7.3 Estatísticas por porte

Além da quantidade, analisamos média, mediana, desvio padrão, mínimo, máximo e quantidade de valores distintos. Um grupo com pouca variabilidade não permite avaliar adequadamente regressão ou R².

In [ ]:
estatisticas_portes = (
    df_atributos
    .groupby("porte_avaliacao", observed=True)["audience"]
    .agg(
        quantidade="size",
        media="mean",
        mediana="median",
        desvio_padrao="std",
        minimo="min",
        maximo="max",
        valores_distintos="nunique",
    )
)
display(estatisticas_portes.round(2))

### 7.4 Visualização do desequilíbrio

O gráfico de contagem evidencia a diferença no número de exemplos. O boxplot em escala logarítmica mostra a dispersão interna sem ocultar os eventos menores.

In [ ]:
PASTA_PORTES = PASTA_PROJETO / "outputs" / "diagnostico_portes"
PASTA_PORTES.mkdir(parents=True, exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(
    data=df_atributos,
    x="porte_avaliacao",
    order=df_atributos["porte_avaliacao"].cat.categories,
    ax=eixos[0],
)
eixos[0].set(title="Quantidade de eventos por porte proposto", xlabel="Porte", ylabel="Eventos")

sns.boxplot(
    data=df_atributos,
    x="porte_avaliacao",
    y="audience",
    order=df_atributos["porte_avaliacao"].cat.categories,
    ax=eixos[1],
)
eixos[1].set(yscale="log", title="Distribuição do público por porte", xlabel="Porte", ylabel="Público — escala log")

fig.tight_layout()
fig.savefig(PASTA_PORTES / "01_distribuicao_faixas_avaliacao.png", dpi=180, bbox_inches="tight")
plt.show()

### 7.5 Critérios mínimos de amostra

Definimos regras antes de observar métricas futuras, evitando escolher critérios apenas para favorecer um resultado:

- menos de 10 eventos: não apresentar R² isolado;
- menos de 30 eventos: não treinar modelo exclusivo para o segmento;
- grupos pequenos podem receber MAE/MAPE descritivos, sempre com ressalva;
- decisões finais devem usar validação cruzada e dispersão, não somente média.

Os limites 10 e 30 são regras conservadoras do projeto, não garantias matemáticas universais. Sua função é impedir conclusões fortes com amostras evidentemente insuficientes.

In [ ]:
MINIMO_R2 = 10
MINIMO_MODELO_SEGMENTADO = 30

avaliacao_suficiencia = estatisticas_portes[["quantidade", "valores_distintos"]].copy()
avaliacao_suficiencia["status_R2"] = np.select(
    [
        avaliacao_suficiencia["quantidade"] < MINIMO_R2,
        avaliacao_suficiencia["quantidade"] < MINIMO_MODELO_SEGMENTADO,
    ],
    ["Não calcular", "Somente exploratório"],
    default="Comparação mais estável",
)
avaliacao_suficiencia["permite_modelo_exclusivo"] = (
    avaliacao_suficiencia["quantidade"] >= MINIMO_MODELO_SEGMENTADO
)
display(avaliacao_suficiencia)

### 7.6 Quartis analíticos

Também criamos faixas baseadas nos quartis da distribuição. Elas não recebem nomes comerciais, pois representam posições relativas dentro desta amostra.

Empates são preservados: eventos com o mesmo público permanecem no mesmo intervalo. Por isso os grupos não precisam ter exatamente o mesmo tamanho. Essa escolha evita separar artificialmente eventos iguais apenas para equilibrar contagens.

In [ ]:
df_atributos["faixa_analitica"] = pd.qcut(
    df_atributos["audience"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop",
)

intervalos_quartis = pd.qcut(
    df_atributos["audience"],
    q=4,
    duplicates="drop",
)

resumo_quartis = (
    df_atributos
    .groupby("faixa_analitica", observed=True)["audience"]
    .agg(quantidade="size", minimo="min", maximo="max", mediana="median")
)
display(resumo_quartis)
print("Intervalos calculados:")
print(intervalos_quartis.cat.categories)

### 7.7 Por que o treinamento usa um único modelo

A estratégia de modelos separados possui dois problemas:

1. **Amostra insuficiente:** mesmo consolidado, Grande/Mega possui apenas 16 eventos;
2. **Roteamento impossível:** para escolher o modelo correto seria necessário conhecer o público real antes da previsão.

Treinar modelos usando o porte verdadeiro seria um experimento-oráculo: pode ser útil como limite teórico, mas não representa um sistema implantável.

A estratégia oficial será um único modelo treinado com os 204 eventos e avaliado separadamente nas três faixas. Grande/Mega receberá métricas exploratórias acompanhadas de `N = 16`. Modelos segmentados só serão reconsiderados futuramente se houver um atributo pré-evento confiável para estimar o porte e uma base muito maior em cada segmento.

### 7.8 Exportação do diagnóstico

As tabelas são exportadas para garantir rastreabilidade e permitir reutilização no relatório e na apresentação.

In [ ]:
estatisticas_portes.to_csv(
    PASTA_PORTES / "estatisticas_faixas_avaliacao.csv",
    encoding="utf-8-sig",
)
avaliacao_suficiencia.to_csv(
    PASTA_PORTES / "suficiencia_amostral.csv",
    encoding="utf-8-sig",
)
resumo_quartis.to_csv(
    PASTA_PORTES / "resumo_quartis.csv",
    encoding="utf-8-sig",
)

print(f"Diagnóstico por porte salvo em: {PASTA_PORTES}")

## 8. Feature Selection

A seleção de atributos busca reduzir dimensionalidade, eliminar redundâncias, melhorar interpretabilidade e limitar overfitting.

Como os métodos possuem vieses diferentes, utilizamos abordagens complementares:

- **filtro:** correlação;
- **embutido em árvores:** importância por redução de impureza;
- **permutação:** queda de desempenho após embaralhar um atributo;
- **regularização:** Lasso;
- **wrapper:** RFECV.

Toda a seleção é realizada somente no conjunto de treino definido no diagnóstico inicial. O conjunto de teste permanece fora desta etapa para evitar escolher atributos com base no resultado que deveria avaliar generalização.

### 8.1 Dados utilizados na seleção

O alvo continua em escala logarítmica, coerente com o modelo inicial e com a assimetria identificada. Começamos pelo cenário A, pois ele contém todos os atributos candidatos. O cenário B será obtido removendo as variáveis de impacto dos subconjuntos selecionados.

In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

indices_treino_fs = X_treino.index
X_fs = df_atributos.loc[indices_treino_fs, atributos_cenario_a].copy()
y_fs = np.log1p(df_atributos.loc[indices_treino_fs, "audience"])

print(f"Observações usadas na seleção: {len(X_fs)}")
print(f"Atributos candidatos: {X_fs.shape[1]}")
assert set(indices_treino_fs).isdisjoint(X_teste.index), "O teste entrou na seleção de atributos."

### 8.2 Correlação com o alvo

Correlação é um método de filtro rápido e interpretável. Usamos Pearson e Spearman porque o primeiro mede associação linear e o segundo associação monotônica.

Correlação baixa não elimina automaticamente um atributo: árvores podem capturar relações não lineares e interações. Correlação alta entre dois atributos também pode indicar redundância, como impacto bruto e impacto logarítmico.

In [ ]:
correlacoes_fs = pd.DataFrame({
    "pearson": X_fs.apply(lambda coluna: coluna.corr(y_fs, method="pearson")),
    "spearman": X_fs.apply(lambda coluna: coluna.corr(y_fs, method="spearman")),
})
correlacoes_fs["maior_abs"] = correlacoes_fs.abs().max(axis=1)
correlacoes_fs = correlacoes_fs.sort_values("maior_abs", ascending=False)

display(correlacoes_fs.round(3))

### 8.3 Redundância entre os atributos

Além da relação com o alvo, verificamos correlação entre as próprias variáveis. Pares muito correlacionados podem carregar informação repetida e prejudicar a interpretação, especialmente em modelos lineares.

Não removemos pares automaticamente: primeiro registramos a redundância e deixamos que os outros métodos indiquem qual representação preservar.

In [ ]:
corr_entre_atributos = X_fs.corr(method="spearman").abs()
triangulo_superior = corr_entre_atributos.where(
    np.triu(np.ones(corr_entre_atributos.shape), k=1).astype(bool)
)

pares_redundantes = (
    triangulo_superior.stack()
    .rename("correlacao_abs")
    .reset_index()
    .rename(columns={"level_0": "atributo_1", "level_1": "atributo_2"})
    .query("correlacao_abs >= 0.80")
    .sort_values("correlacao_abs", ascending=False)
)

display(pares_redundantes.round(3))

### 8.4 Importância embutida do Random Forest

O Random Forest fornece `feature_importances_`, baseada na redução de impureza produzida pelos atributos nas árvores.

Vantagem: captura relações não lineares. Limitação: pode favorecer variáveis contínuas ou com mais possibilidades de divisão. Por isso ela será comparada à importância por permutação.

In [ ]:
rf_importancia = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)
rf_importancia.fit(X_fs, y_fs)

importancia_impureza = pd.Series(
    rf_importancia.feature_importances_,
    index=X_fs.columns,
    name="importancia_impureza",
).sort_values(ascending=False)

display(importancia_impureza.to_frame().round(4))

### 8.5 Importância por permutação

Para não medir a permutação nos mesmos dados usados no ajuste, o conjunto de treino é dividido novamente em subtreino e validação. O teste principal continua intocado.

Ao embaralhar um atributo, quebramos sua relação com o alvo. Se o MAE em log piorar, o atributo contém informação útil para aquele modelo. Importâncias próximas de zero ou negativas indicam pouca contribuição nessa validação específica.

In [ ]:
X_subtreino, X_validacao, y_subtreino, y_validacao = train_test_split(
    X_fs,
    y_fs,
    test_size=0.25,
    random_state=SEED,
)

rf_permutacao = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)
rf_permutacao.fit(X_subtreino, y_subtreino)

resultado_permutacao = permutation_importance(
    rf_permutacao,
    X_validacao,
    y_validacao,
    scoring="neg_mean_absolute_error",
    n_repeats=30,
    random_state=SEED,
    n_jobs=-1,
)

importancia_permutacao = pd.DataFrame({
    "atributo": X_fs.columns,
    "importancia_media": resultado_permutacao.importances_mean,
    "desvio_padrao": resultado_permutacao.importances_std,
}).sort_values("importancia_media", ascending=False)

display(importancia_permutacao.round(4))

### 8.6 Visualização das importâncias

As duas importâncias são exibidas lado a lado, mas suas escalas não são diretamente comparáveis. O objetivo é comparar a ordem e identificar atributos que parecem importantes apenas pelo critério interno das árvores.

In [ ]:
PASTA_FS = PASTA_PROJETO / "outputs" / "feature_selection"
PASTA_FS.mkdir(parents=True, exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(15, 7))

importancia_impureza.sort_values().plot(kind="barh", ax=eixos[0])
eixos[0].set(title="Importância por redução de impureza", xlabel="Importância", ylabel="")

permutacao_plot = importancia_permutacao.sort_values("importancia_media")
eixos[1].barh(
    permutacao_plot["atributo"],
    permutacao_plot["importancia_media"],
    xerr=permutacao_plot["desvio_padrao"],
)
eixos[1].axvline(0, color="black", linewidth=1)
eixos[1].set(title="Importância por permutação", xlabel="Aumento do desempenho ao preservar o atributo", ylabel="")

fig.tight_layout()
fig.savefig(PASTA_FS / "01_importancias_random_forest.png", dpi=180, bbox_inches="tight")
plt.show()

### 8.7 Regularização Lasso

Lasso adiciona uma penalização L1 ao modelo linear e pode reduzir coeficientes a zero. Antes do ajuste, todos os atributos são padronizados, pois a penalização depende da escala.

Vantagens: seleção embutida e resultado interpretável. Limitações: pressupõe relação linear e pode escolher apenas uma variável entre atributos fortemente correlacionados. `LassoCV` escolhe a intensidade da penalização por validação cruzada dentro do conjunto de treino.

In [ ]:
cv_fs = KFold(n_splits=5, shuffle=True, random_state=SEED)

pipeline_lasso = Pipeline([
    ("escala", StandardScaler()),
    ("lasso", LassoCV(cv=cv_fs, random_state=SEED, max_iter=50_000)),
])
pipeline_lasso.fit(X_fs, y_fs)

coeficientes_lasso = pd.Series(
    pipeline_lasso.named_steps["lasso"].coef_,
    index=X_fs.columns,
    name="coeficiente",
)
atributos_lasso = coeficientes_lasso[coeficientes_lasso.abs() > 1e-8].index.tolist()

print(f"Alpha escolhido: {pipeline_lasso.named_steps['lasso'].alpha_:.6f}")
print(f"Atributos com coeficiente diferente de zero: {len(atributos_lasso)}")
display(coeficientes_lasso.reindex(coeficientes_lasso.abs().sort_values(ascending=False).index).to_frame().round(4))

### 8.8 RFECV — eliminação recursiva com validação cruzada

O RFECV remove progressivamente os atributos menos importantes e avalia os subconjuntos por validação cruzada. Utilizamos Random Forest para permitir relações não lineares e MAE negativo em escala log como critério.

O método é computacionalmente mais caro que correlação e Lasso, pois treina diversos modelos. Esse custo será considerado na análise de viabilidade. O resultado ainda é um candidato, não uma decisão final.

In [ ]:
estimador_rfe = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)

seletor_rfecv = RFECV(
    estimator=estimador_rfe,
    step=1,
    min_features_to_select=3,
    cv=cv_fs,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
seletor_rfecv.fit(X_fs, y_fs)

resultado_rfe = pd.DataFrame({
    "atributo": X_fs.columns,
    "selecionado": seletor_rfecv.support_,
    "ranking": seletor_rfecv.ranking_,
}).sort_values(["selecionado", "ranking"], ascending=[False, True])

atributos_rfe = resultado_rfe.loc[resultado_rfe["selecionado"], "atributo"].tolist()
print(f"Quantidade selecionada pelo RFECV: {seletor_rfecv.n_features_}")
display(resultado_rfe)

### 8.9 Consenso entre os métodos

Criamos uma tabela de votos com três métodos que efetivamente selecionam atributos:

- permutação acima de seu desvio padrão;
- coeficiente Lasso diferente de zero;
- seleção pelo RFECV.

Um atributo entra no conjunto de consenso quando recebe pelo menos dois votos. Esse limite é transparente e evita depender de uma única técnica. O conjunto de consenso ainda será comparado aos demais conjuntos; ele não é automaticamente o conjunto final.

In [ ]:
atributos_permutacao = importancia_permutacao.loc[
    (importancia_permutacao["importancia_media"] > importancia_permutacao["desvio_padrao"])
    & (importancia_permutacao["importancia_media"] > 0.001),
    "atributo",
].tolist()

tabela_consenso = pd.DataFrame(index=X_fs.columns)
tabela_consenso["permutacao_consistente"] = tabela_consenso.index.isin(atributos_permutacao).astype(int)
tabela_consenso["lasso"] = tabela_consenso.index.isin(atributos_lasso).astype(int)
tabela_consenso["rfecv"] = tabela_consenso.index.isin(atributos_rfe).astype(int)
tabela_consenso["votos"] = tabela_consenso.sum(axis=1)
tabela_consenso = tabela_consenso.sort_values("votos", ascending=False)

atributos_consenso_a = tabela_consenso.index[tabela_consenso["votos"] >= 2].tolist()
atributos_consenso_b = [
    atributo for atributo in atributos_consenso_a
    if atributo not in {"impacto_milhoes_nominais", "log_impacto_nominal"}
]

display(tabela_consenso)
print("Consenso — cenário A:", atributos_consenso_a)
print("Consenso — cenário B:", atributos_consenso_b)

### 8.10 Votos do consenso

O gráfico torna explícito onde os métodos concordam. Votos não são medidas de efeito nem garantem causalidade; eles apenas resumem quantos dos três procedimentos selecionaram cada atributo.

In [ ]:
consenso_plot = tabela_consenso.reset_index(names="atributo").sort_values(["votos", "atributo"])
fig, ax = plt.subplots(figsize=(9, 7))
cores_consenso = consenso_plot["votos"].map({0: "#d9d9d9", 1: "#9ecae1", 2: "#4292c6", 3: "#08519c"})
ax.barh(consenso_plot["atributo"], consenso_plot["votos"], color=cores_consenso)
ax.axvline(2, color="black", linestyle="--", linewidth=1, label="limite do consenso")
ax.set(title="Consenso entre Permutação, Lasso e RFECV", xlabel="Quantidade de votos", ylabel="")
ax.set_xticks([0, 1, 2, 3])
ax.legend()
fig.tight_layout()
fig.savefig(PASTA_FS / "02_votos_consenso.png", dpi=180, bbox_inches="tight")
plt.show()

### 8.11 Redução de redundância no consenso

Os métodos automáticos selecionaram simultaneamente atributos fortemente correlacionados. Aplicamos uma regra técnica explícita:

- entre impacto bruto e logarítmico, preservamos o log, que possui menor assimetria e relação linear mais forte com o alvo log;
- entre idade central e amplitude, preservamos amplitude, que teve importância por permutação ligeiramente maior e representa a abrangência etária.

Essa decisão reduz dimensionalidade e melhora interpretação. Como diferenças pequenas podem depender da amostra, os conjuntos não reduzidos continuam disponíveis para comparação.

In [ ]:
atributos_consenso_reduzido_a = [
    atributo for atributo in atributos_consenso_a
    if atributo not in {"impacto_milhoes_nominais", "idade_central"}
]
atributos_consenso_reduzido_b = [
    atributo for atributo in atributos_consenso_b
    if atributo != "idade_central"
]

print("Consenso reduzido — cenário A:", atributos_consenso_reduzido_a)
print("Consenso reduzido — cenário B:", atributos_consenso_reduzido_b)

### 8.11 Conjuntos que serão comparados

Para medir se a seleção realmente agrega valor, serão comparados posteriormente:

1. impacto nominal — referência do Trabalho 1;
2. impacto logarítmico;
3. todos os atributos do cenário A;
4. atributos selecionados pelo RFECV;
5. atributos de consenso do cenário A;
6. consenso reduzido do cenário A;
7. todos os atributos pré-evento do cenário B;
8. consenso pré-evento do cenário B;
9. consenso reduzido do cenário B.

Definir os conjuntos antes da comparação reduz o risco de escolher apenas a combinação que apresentar o melhor resultado por acaso.

In [ ]:
conjuntos_atributos = {
    "impacto_original": ["impacto_milhoes_nominais"],
    "impacto_log": ["log_impacto_nominal"],
    "todos_cenario_a": atributos_cenario_a,
    "rfecv_cenario_a": atributos_rfe,
    "consenso_cenario_a": atributos_consenso_a,
    "consenso_reduzido_a": atributos_consenso_reduzido_a,
    "todos_cenario_b": atributos_cenario_b,
    "consenso_cenario_b": atributos_consenso_b,
    "consenso_reduzido_b": atributos_consenso_reduzido_b,
}

# Um conjunto vazio não pode ser modelado. Caso nenhum atributo pré-evento
# alcance dois votos, preservamos todos os atributos do cenário B como candidato.
if not conjuntos_atributos["consenso_cenario_b"]:
    conjuntos_atributos["consenso_cenario_b"] = atributos_cenario_b.copy()
    print("Nenhum atributo pré-evento atingiu dois votos; o cenário B completo foi preservado.")

resumo_conjuntos = pd.DataFrame([
    {"conjunto": nome, "quantidade": len(atributos), "atributos": ", ".join(atributos)}
    for nome, atributos in conjuntos_atributos.items()
])
display(resumo_conjuntos)

### 8.12 Exportação dos resultados

As tabelas são salvas para permitir auditoria, uso no relatório e repetição das comparações.

In [ ]:
correlacoes_fs.to_csv(PASTA_FS / "correlacoes_com_alvo.csv", encoding="utf-8-sig")
pares_redundantes.to_csv(PASTA_FS / "pares_redundantes.csv", index=False, encoding="utf-8-sig")
importancia_impureza.to_csv(PASTA_FS / "importancia_impureza.csv", encoding="utf-8-sig")
importancia_permutacao.to_csv(PASTA_FS / "importancia_permutacao.csv", index=False, encoding="utf-8-sig")
coeficientes_lasso.to_csv(PASTA_FS / "coeficientes_lasso.csv", encoding="utf-8-sig")
resultado_rfe.to_csv(PASTA_FS / "resultado_rfecv.csv", index=False, encoding="utf-8-sig")
tabela_consenso.to_csv(PASTA_FS / "consenso_metodos.csv", encoding="utf-8-sig")
resumo_conjuntos.to_csv(PASTA_FS / "conjuntos_candidatos.csv", index=False, encoding="utf-8-sig")

print(f"Resultados de feature selection salvos em: {PASTA_FS}")

## 9. Comparação de modelos

Nesta etapa comparamos algoritmos sem fine tuning. O objetivo é identificar quais famílias de modelos merecem ajuste posterior, mantendo a comparação justa.

Todos os modelos usam:

- os mesmos cinco folds;
- alvo `log1p(audience)` durante o treinamento;
- métricas calculadas após retornar as previsões para número de pessoas;
- configurações padrão ou equivalentes à referência do Trabalho 1.

Os cenários A e B são analisados separadamente. O cenário A mede desempenho acadêmico com impacto; o cenário B mede o potencial das informações pré-evento disponíveis.

### 9.1 Algoritmos comparados

**Regressão Linear:** baseline paramétrico interpretável e de baixo custo, mas limitado a relações lineares.

**Árvore de Decisão:** captura não linearidade e é explicável, porém pode apresentar alta variância e overfitting.

**Random Forest:** reduz a variância combinando várias árvores e foi o algoritmo do Trabalho 1. Possui maior custo e menor transparência que uma árvore individual.

**Gradient Boosting:** constrói árvores sequencialmente para corrigir erros anteriores. Pode alcançar bom desempenho em dados tabulares, mas é mais sensível a hiperparâmetros.

XGBoost foi adiado porque exigiria uma dependência adicional e seu ganho não é garantido em uma base de 204 registros. A prioridade é comparar métodos já suficientes para demonstrar refinamento técnico.

In [ ]:
import time

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

modelos_base = {
    "Regressão Linear": Pipeline([
        ("escala", StandardScaler()),
        ("modelo", LinearRegression()),
    ]),
    "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=SEED,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
}

cv_comparacao = KFold(n_splits=5, shuffle=True, random_state=SEED)
print("Modelos:", list(modelos_base))

### 9.2 Conjuntos selecionados para a comparação

Para evitar 36 combinações pouco informativas, usamos cinco conjuntos representativos:

- impacto original — referência direta do Trabalho 1;
- impacto log — transformação mais adequada à assimetria;
- todos do cenário A — verifica se adicionar todas as variáveis ajuda;
- consenso reduzido A — conjunto compacto com impacto;
- todos do cenário B e consenso reduzido B — avaliam implantação pré-evento.

Os conjuntos RFECV e consenso completo continuam exportados, mas os reduzidos são priorizados por menor redundância e maior interpretabilidade.

In [ ]:
conjuntos_comparacao = {
    "A — impacto original": conjuntos_atributos["impacto_original"],
    "A — impacto log": conjuntos_atributos["impacto_log"],
    "A — todos": conjuntos_atributos["todos_cenario_a"],
    "A — consenso reduzido": conjuntos_atributos["consenso_reduzido_a"],
    "B — todos pré-evento": conjuntos_atributos["todos_cenario_b"],
    "B — consenso reduzido": conjuntos_atributos["consenso_reduzido_b"],
}

display(pd.DataFrame([
    {"conjunto": nome, "quantidade": len(atributos), "atributos": ", ".join(atributos)}
    for nome, atributos in conjuntos_comparacao.items()
]))

### 9.3 Procedimento de avaliação

O modelo é treinado cinco vezes. Em cada fold, aproximadamente 80% dos dados são usados para treino e 20% para validação. Cada evento recebe uma previsão fora do fold em que foi treinado.

Registramos:

- métricas globais sobre todas as previsões fora da amostra;
- média e desvio do MAE entre folds;
- tempo médio de treinamento;
- tempo médio de inferência por evento.

O MAE é a métrica principal por ser diretamente interpretável. RMSE e WAPE são critérios secundários. R², MAPE, estabilidade e custo computacional completam a decisão.

In [ ]:
def avaliar_modelo_cv(nome_modelo, estimador, nome_conjunto, atributos):
    X = df_atributos[atributos]
    y = df_atributos["audience"].to_numpy()
    y_transformado = np.log1p(y)
    previsoes_oof = np.zeros(len(df_atributos), dtype=float)
    maes_folds = []
    tempos_treino = []
    tempos_inferencia = []

    for indices_treino, indices_validacao in cv_comparacao.split(X):
        modelo = clone(estimador)

        inicio_treino = time.perf_counter()
        modelo.fit(X.iloc[indices_treino], y_transformado[indices_treino])
        tempos_treino.append(time.perf_counter() - inicio_treino)

        inicio_inferencia = time.perf_counter()
        previsao_log = modelo.predict(X.iloc[indices_validacao])
        tempo_inferencia = time.perf_counter() - inicio_inferencia
        tempos_inferencia.append(tempo_inferencia / len(indices_validacao))

        previsao_real = np.maximum(0, np.expm1(previsao_log))
        previsoes_oof[indices_validacao] = previsao_real
        maes_folds.append(mean_absolute_error(y[indices_validacao], previsao_real))

    metricas = calcular_metricas(y, previsoes_oof)
    return {
        "cenario": nome_conjunto.split(" — ")[0],
        "conjunto": nome_conjunto,
        "modelo": nome_modelo,
        "n_atributos": len(atributos),
        **metricas,
        "MAE_fold_media": np.mean(maes_folds),
        "MAE_fold_dp": np.std(maes_folds, ddof=1),
        "tempo_treino_medio_s": np.mean(tempos_treino),
        "inferencia_media_ms_evento": np.mean(tempos_inferencia) * 1000,
    }, previsoes_oof

### 9.4 Execução da comparação

O mesmo procedimento é aplicado a todas as combinações. As previsões fora da amostra são preservadas para a posterior avaliação por porte e análise de erros.

In [ ]:
resultados_modelos = []
previsoes_comparacao = {}

for nome_conjunto, atributos in conjuntos_comparacao.items():
    for nome_modelo, estimador in modelos_base.items():
        resultado, previsoes = avaliar_modelo_cv(
            nome_modelo,
            estimador,
            nome_conjunto,
            atributos,
        )
        resultados_modelos.append(resultado)
        previsoes_comparacao[(nome_conjunto, nome_modelo)] = previsoes

tabela_modelos = pd.DataFrame(resultados_modelos)
tabela_modelos = tabela_modelos.sort_values(["cenario", "MAE", "RMSE"]).reset_index(drop=True)

display(tabela_modelos.round(3))

### 9.5 Ranking separado por cenário

Não criamos um único ranking misturando A e B. O cenário A pode ter melhor métrica porque utiliza impacto, mas isso não o torna automaticamente implantável. O cenário B possui uma pergunta de negócio diferente: quanto conseguimos prever apenas com os atributos disponíveis antes do evento nesta base?

In [ ]:
melhores_por_cenario = (
    tabela_modelos
    .sort_values(["cenario", "MAE", "RMSE"])
    .groupby("cenario", as_index=False)
    .first()
)

display(melhores_por_cenario[[
    "cenario", "conjunto", "modelo", "n_atributos",
    "MAE", "RMSE", "R2", "MAPE_%", "WAPE_%",
    "MAE_fold_dp", "tempo_treino_medio_s",
]].round(3))

### 9.6 Visualização desempenho versus estabilidade

O gráfico mostra MAE médio e desvio entre folds. Um modelo com média ligeiramente melhor, mas grande variabilidade, pode ser menos confiável que uma alternativa estável.

In [ ]:
PASTA_MODELOS = PASTA_PROJETO / "outputs" / "comparacao_modelos"
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(16, 7))

for eixo, cenario in zip(eixos, ["A", "B"]):
    dados_plot = tabela_modelos[tabela_modelos["cenario"] == cenario].copy()
    dados_plot["combinacao"] = dados_plot["modelo"] + "\n" + dados_plot["conjunto"].str.replace(f"{cenario} — ", "", regex=False)
    dados_plot = dados_plot.sort_values("MAE", ascending=True)
    eixo.barh(
        dados_plot["combinacao"],
        dados_plot["MAE"],
        xerr=dados_plot["MAE_fold_dp"],
        alpha=0.8,
    )
    eixo.set(title=f"Cenário {cenario}: MAE e variação entre folds", xlabel="MAE em pessoas", ylabel="")

fig.tight_layout()
fig.savefig(PASTA_MODELOS / "01_mae_estabilidade_por_cenario.png", dpi=180, bbox_inches="tight")
plt.show()

### 9.7 Comparação compacta dos algoritmos no cenário A

Para não repetir todas as combinações, selecionamos aqui a melhor configuração de atributos de cada algoritmo segundo o MAE do cenário A. MAE e RMSE continuam em pessoas; o tempo de treino possui escala própria e é exibido separadamente.

In [ ]:
melhor_a_por_algoritmo = (
    tabela_modelos.query("cenario == 'A'")
    .sort_values(["modelo", "MAE", "RMSE"])
    .groupby("modelo", as_index=False)
    .first()
    .sort_values("MAE")
)

fig, eixos = plt.subplots(1, 3, figsize=(17, 5))
sns.barplot(data=melhor_a_por_algoritmo, x="modelo", y="MAE", ax=eixos[0])
eixos[0].set(title="MAE — melhor conjunto por algoritmo", xlabel="", ylabel="Pessoas")
sns.barplot(data=melhor_a_por_algoritmo, x="modelo", y="RMSE", ax=eixos[1])
eixos[1].set(title="RMSE — melhor conjunto por algoritmo", xlabel="", ylabel="Pessoas")
sns.barplot(data=melhor_a_por_algoritmo, x="modelo", y="tempo_treino_medio_s", ax=eixos[2])
eixos[2].set(title="Tempo médio de treinamento", xlabel="", ylabel="Segundos")
for eixo in eixos:
    eixo.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(PASTA_MODELOS / "02_comparacao_compacta_algoritmos.png", dpi=180, bbox_inches="tight")
plt.show()

### 9.8 Critério para encaminhar modelos ao fine tuning

O fine tuning será realizado apenas nos modelos de árvore/ensemble mais competitivos do cenário A. Ajustar todos os algoritmos aumentaria o custo e o risco de encontrar uma configuração favorável por acaso.

A Regressão Linear permanece como referência interpretável. O cenário B não será “otimizado até parecer bom”: seu resultado servirá principalmente para demonstrar a limitação dos dados pré-evento atuais e orientar a coleta futura.

In [ ]:
candidatos_nao_lineares_a = tabela_modelos[
    (tabela_modelos["cenario"] == "A")
    & (tabela_modelos["modelo"].isin(["Árvore de Decisão", "Random Forest", "Gradient Boosting"]))
].sort_values(["MAE", "RMSE"])

modelos_para_tuning = candidatos_nao_lineares_a["modelo"].drop_duplicates().head(2).tolist()
if "Random Forest" not in modelos_para_tuning:
    modelos_para_tuning.append("Random Forest")
melhor_conjunto_a = melhores_por_cenario.loc[
    melhores_por_cenario["cenario"] == "A", "conjunto"
].iloc[0]

print("Modelos encaminhados ao fine tuning:", modelos_para_tuning)
print("Melhor conjunto preliminar do cenário A:", melhor_conjunto_a)

### 9.8 Exportação

A tabela completa e as previsões fora da amostra são exportadas. Isso permite reutilizar exatamente os mesmos resultados na avaliação por porte, sem treinar o modelo novamente com divisões diferentes.

In [ ]:
tabela_modelos.to_csv(
    PASTA_MODELOS / "comparacao_modelos_cv.csv",
    index=False,
    encoding="utf-8-sig",
)

previsoes_exportar = pd.DataFrame({"audience_real": df_atributos["audience"]})
for (conjunto, modelo), previsoes in previsoes_comparacao.items():
    nome_coluna = f"{conjunto} | {modelo}".replace(" — ", "_").replace(" ", "_")
    previsoes_exportar[nome_coluna] = previsoes

previsoes_exportar.to_csv(
    PASTA_MODELOS / "previsoes_fora_da_amostra.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Resultados da comparação salvos em: {PASTA_MODELOS}")

## 10. Fine tuning

O fine tuning busca configurações que controlem complexidade, reduzam overfitting e melhorem generalização. Ele é aplicado somente aos modelos não lineares competitivos e ao Random Forest do Trabalho 1.

Uma grade excessivamente grande aumentaria custo e risco de selecionar uma configuração favorável por acaso. Portanto, testamos valores moderados e tecnicamente interpretáveis.

A Regressão Linear não participa do GridSearchCV porque não possui hiperparâmetros equivalentes de complexidade nessa implementação. Ela permanece como candidata de referência.

### 10.1 Conjunto de atributos por modelo

Cada algoritmo é ajustado com o conjunto que apresentou seu menor MAE na comparação anterior. Isso evita ajustar um modelo com uma representação em que ele já se mostrou inadequado.

Os conjuntos são fixados antes da busca. O fine tuning altera apenas hiperparâmetros, não volta a escolher atributos.

In [ ]:
melhor_linha_por_modelo_a = (
    tabela_modelos[
        (tabela_modelos["cenario"] == "A")
        & (tabela_modelos["modelo"].isin(["Árvore de Decisão", "Random Forest", "Gradient Boosting"]))
    ]
    .sort_values(["modelo", "MAE", "RMSE"])
    .groupby("modelo", as_index=False)
    .first()
)

conjunto_por_modelo = {
    linha["modelo"]: linha["conjunto"]
    for _, linha in melhor_linha_por_modelo_a.iterrows()
}

display(melhor_linha_por_modelo_a[["modelo", "conjunto", "MAE", "RMSE", "R2"]].round(3))

### 10.2 Métrica de otimização na escala real

Os modelos continuam treinando `log1p(audience)`, mas o GridSearchCV deve escolher parâmetros com base no erro em pessoas. Criamos um scorer que aplica `expm1` antes de calcular MAE.

Essa decisão evita escolher uma configuração excelente na escala log, mas inadequada para o erro absoluto usado no negócio.

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, cross_val_predict

def mae_real_partindo_de_log(y_log_verdadeiro, y_log_previsto):
    y_real_local = np.expm1(np.asarray(y_log_verdadeiro))
    previsao_real_local = np.maximum(0, np.expm1(np.asarray(y_log_previsto)))
    return mean_absolute_error(y_real_local, previsao_real_local)

scorer_mae_real = make_scorer(
    mae_real_partindo_de_log,
    greater_is_better=False,
)

### 10.3 Grades de hiperparâmetros

**Árvore de Decisão**

- `max_depth`: limita profundidade;
- `min_samples_split`: exige mais exemplos para dividir um nó;
- `min_samples_leaf`: impede folhas baseadas em poucos eventos;
- `max_features`: limita atributos considerados em cada divisão.

**Random Forest**

- `n_estimators`: quantidade de árvores;
- `max_depth`, `min_samples_split` e `min_samples_leaf`: controlam complexidade;
- `max_features`: controla diversidade entre árvores.

**Gradient Boosting**

- `n_estimators`: quantidade de etapas sequenciais;
- `learning_rate`: contribuição de cada etapa;
- `max_depth`: complexidade das árvores fracas;
- `min_samples_leaf`: regularização das folhas;
- `subsample`: usa parte dos dados em cada etapa e pode reduzir variância.

In [ ]:
configuracoes_tuning = {
    "Árvore de Decisão": {
        "estimador": DecisionTreeRegressor(random_state=SEED),
        "grade": {
            "max_depth": [3, 5, 8, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": [None, "sqrt"],
        },
    },
    "Random Forest": {
        "estimador": RandomForestRegressor(random_state=SEED, n_jobs=-1),
        "grade": {
            "n_estimators": [100, 300],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4],
            "max_features": [1.0, "sqrt"],
        },
    },
    "Gradient Boosting": {
        "estimador": GradientBoostingRegressor(random_state=SEED),
        "grade": {
            "n_estimators": [50, 100],
            "learning_rate": [0.03, 0.05, 0.10],
            "max_depth": [2, 3],
            "min_samples_leaf": [1, 3],
            "subsample": [0.8, 1.0],
        },
    },
}

for nome, configuracao in configuracoes_tuning.items():
    combinacoes = int(np.prod([len(valores) for valores in configuracao["grade"].values()]))
    print(f"{nome}: {combinacoes} combinações × 5 folds")

### 10.4 Execução do GridSearchCV

O GridSearchCV testa todas as combinações usando os mesmos cinco folds. Registramos tempo total, melhores parâmetros e melhor MAE interno.

O custo é aceitável para esta prova de conceito, mas o crescimento da grade é multiplicativo. Esse comportamento será considerado na viabilidade técnica.

In [ ]:
buscas_tuning = {}
resumo_tuning = []

for nome_modelo, configuracao in configuracoes_tuning.items():
    nome_conjunto = conjunto_por_modelo[nome_modelo]
    atributos = conjuntos_comparacao[nome_conjunto]
    X_tuning = df_atributos[atributos]
    y_tuning = np.log1p(df_atributos["audience"])

    busca = GridSearchCV(
        estimator=configuracao["estimador"],
        param_grid=configuracao["grade"],
        scoring=scorer_mae_real,
        cv=cv_comparacao,
        n_jobs=-1,
        return_train_score=True,
        refit=True,
    )

    inicio = time.perf_counter()
    busca.fit(X_tuning, y_tuning)
    tempo_total = time.perf_counter() - inicio

    buscas_tuning[nome_modelo] = busca
    resumo_tuning.append({
        "modelo": nome_modelo,
        "conjunto": nome_conjunto,
        "n_atributos": len(atributos),
        "melhor_MAE_cv_interno": -busca.best_score_,
        "tempo_busca_s": tempo_total,
        "melhores_parametros": busca.best_params_,
    })

tabela_tuning = pd.DataFrame(resumo_tuning).sort_values("melhor_MAE_cv_interno")
display(tabela_tuning)

### 10.5 Comparação antes e depois do ajuste

Geramos previsões fora da amostra com o melhor estimador de cada busca. Como os parâmetros foram escolhidos usando esses mesmos folds, o resultado ajustado é preliminar e pode ser otimista.

A comparação formal será repetida com validação aninhada ou repetida na etapa seguinte.

In [ ]:
comparacao_tuning = []
previsoes_tunadas = {}

for nome_modelo, busca in buscas_tuning.items():
    nome_conjunto = conjunto_por_modelo[nome_modelo]
    atributos = conjuntos_comparacao[nome_conjunto]
    X_tuning = df_atributos[atributos]
    y_real_tuning = df_atributos["audience"].to_numpy()
    y_log_tuning = np.log1p(y_real_tuning)

    previsoes_log = cross_val_predict(
        clone(busca.best_estimator_),
        X_tuning,
        y_log_tuning,
        cv=cv_comparacao,
        n_jobs=-1,
    )
    previsoes_reais = np.maximum(0, np.expm1(previsoes_log))
    previsoes_tunadas[nome_modelo] = previsoes_reais
    metricas_depois = calcular_metricas(y_real_tuning, previsoes_reais)

    linha_antes = tabela_modelos[
        (tabela_modelos["modelo"] == nome_modelo)
        & (tabela_modelos["conjunto"] == nome_conjunto)
    ].iloc[0]

    comparacao_tuning.append({
        "modelo": nome_modelo,
        "conjunto": nome_conjunto,
        "MAE_antes": linha_antes["MAE"],
        "MAE_depois_preliminar": metricas_depois["MAE"],
        "variacao_MAE_%": (metricas_depois["MAE"] / linha_antes["MAE"] - 1) * 100,
        "RMSE_antes": linha_antes["RMSE"],
        "RMSE_depois_preliminar": metricas_depois["RMSE"],
        "R2_antes": linha_antes["R2"],
        "R2_depois_preliminar": metricas_depois["R2"],
        "MAPE_depois_%": metricas_depois["MAPE_%"],
        "WAPE_depois_%": metricas_depois["WAPE_%"],
    })

tabela_antes_depois = pd.DataFrame(comparacao_tuning).sort_values("MAE_depois_preliminar")
display(tabela_antes_depois.round(3))

### 10.6 Overfitting na melhor configuração

O GridSearchCV também registra a pontuação de treino. Uma diferença grande entre MAE de treino e validação indica que a configuração pode continuar complexa demais.

Essa análise considera somente a melhor linha de cada grade e será complementada pela validação aninhada.

In [ ]:
diagnostico_overfitting_tuning = []

for nome_modelo, busca in buscas_tuning.items():
    melhor_indice = busca.best_index_
    mae_treino_grid = -busca.cv_results_["mean_train_score"][melhor_indice]
    mae_validacao_grid = -busca.cv_results_["mean_test_score"][melhor_indice]
    diagnostico_overfitting_tuning.append({
        "modelo": nome_modelo,
        "MAE_treino_grid": mae_treino_grid,
        "MAE_validacao_grid": mae_validacao_grid,
        "razao_validacao_treino": mae_validacao_grid / mae_treino_grid,
    })

tabela_overfitting_tuning = pd.DataFrame(diagnostico_overfitting_tuning)
display(tabela_overfitting_tuning.round(3))

### 10.7 Visualização da melhoria preliminar

Valores menores de MAE são melhores. O gráfico permite identificar se o custo do GridSearchCV produziu ganho relevante ou apenas alteração marginal.

In [ ]:
PASTA_TUNING = PASTA_PROJETO / "outputs" / "fine_tuning"
PASTA_TUNING.mkdir(parents=True, exist_ok=True)

dados_plot_tuning = tabela_antes_depois.set_index("modelo")[["MAE_antes", "MAE_depois_preliminar"]]
ax = dados_plot_tuning.plot(kind="bar", figsize=(10, 5))
ax.set(title="MAE antes e depois do fine tuning", xlabel="Modelo", ylabel="MAE em pessoas")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(PASTA_TUNING / "01_mae_antes_depois.png", dpi=180, bbox_inches="tight")
plt.show()

### 10.8 Exportação

Além dos resumos, exportamos o resultado completo de cada grade. Isso permite verificar todas as combinações testadas e evita apresentar somente a configuração vencedora.

In [ ]:
tabela_tuning.to_csv(PASTA_TUNING / "resumo_grid_search.csv", index=False, encoding="utf-8-sig")
tabela_antes_depois.to_csv(PASTA_TUNING / "comparacao_antes_depois.csv", index=False, encoding="utf-8-sig")
tabela_overfitting_tuning.to_csv(PASTA_TUNING / "overfitting_grid.csv", index=False, encoding="utf-8-sig")

for nome_modelo, busca in buscas_tuning.items():
    nome_arquivo = (
        nome_modelo.lower()
        .replace(" ", "_")
        .replace("á", "a")
        .replace("ã", "a")
        .replace("ç", "c")
    )
    pd.DataFrame(busca.cv_results_).to_csv(
        PASTA_TUNING / f"grade_completa_{nome_arquivo}.csv",
        index=False,
        encoding="utf-8-sig",
    )

print(f"Resultados do fine tuning salvos em: {PASTA_TUNING}")

## 11. Validação cruzada

O resultado do fine tuning pode ser otimista porque a melhor configuração foi escolhida observando os mesmos folds usados na comparação preliminar.

Para separar ajuste e avaliação, utilizamos validação cruzada aninhada:

- **loop externo com 5 folds:** estima desempenho fora da amostra;
- **loop interno com 4 folds:** escolhe hiperparâmetros do Gradient Boosting usando somente o treino externo.

A Regressão Linear não necessita de busca interna, mas é avaliada nos mesmos folds externos para garantir comparação justa.

### 11.1 Folds externos estratificados por faixa

Em regressão, KFold não garante que os poucos eventos Grande/Mega apareçam em todos os folds. Utilizamos `StratifiedKFold` com `porte_avaliacao` somente para distribuir as escalas entre os folds.

O porte não entra no modelo. Ele controla apenas a composição dos conjuntos de avaliação. Cada fold recebe aproximadamente três eventos Grande/Mega, reduzindo a chance de um fold concentrar todos os maiores eventos.

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv_externo = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_interno = KFold(n_splits=4, shuffle=True, random_state=SEED)

estratos = df_atributos["porte_avaliacao"]
distribuicao_folds = []

for numero_fold, (_, indices_validacao) in enumerate(
    cv_externo.split(df_atributos, estratos), start=1
):
    contagens = estratos.iloc[indices_validacao].value_counts().to_dict()
    distribuicao_folds.append({"fold": numero_fold, **contagens})

display(pd.DataFrame(distribuicao_folds).fillna(0).astype(int))

### 11.2 Candidatos finais da validação

Comparamos os dois candidatos mais fortes e conceitualmente distintos:

- **Regressão Linear:** cinco atributos do consenso reduzido A, baixo custo e maior interpretabilidade;
- **Gradient Boosting:** somente impacto log, melhor resultado após fine tuning.

Árvore e Random Forest não seguem para a validação aninhada porque apresentaram MAE maior, maior custo ou sinal de overfitting. Seus resultados permanecem documentados como comparação.

In [ ]:
atributos_linear_finalista = conjuntos_comparacao["A — consenso reduzido"]
atributos_gb_finalista = conjuntos_comparacao["A — impacto log"]

modelo_linear_finalista = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", LinearRegression()),
])

grade_gb_aninhada = configuracoes_tuning["Gradient Boosting"]["grade"]

print("Regressão Linear:", atributos_linear_finalista)
print("Gradient Boosting:", atributos_gb_finalista)

### 11.3 Função de métricas por fold

As métricas são calculadas separadamente em cada fold externo. Isso permite avaliar não apenas a média, mas também a dispersão.

R² pode variar bastante quando cada fold contém poucos eventos grandes. Por isso, MAE, RMSE, MAPE e WAPE continuam sendo analisados em conjunto.

In [ ]:
def registrar_metricas_fold(modelo, fold, y_treino_real_fold, pred_treino, y_validacao_real_fold, pred_validacao, tempo_s, parametros):
    metricas_treino_fold = calcular_metricas(y_treino_real_fold, pred_treino)
    metricas_validacao_fold = calcular_metricas(y_validacao_real_fold, pred_validacao)
    return {
        "modelo": modelo,
        "fold": fold,
        "N_validacao": len(y_validacao_real_fold),
        "MAE_treino": metricas_treino_fold["MAE"],
        "MAE": metricas_validacao_fold["MAE"],
        "RMSE": metricas_validacao_fold["RMSE"],
        "R2": metricas_validacao_fold["R2"],
        "MAPE_%": metricas_validacao_fold["MAPE_%"],
        "WAPE_%": metricas_validacao_fold["WAPE_%"],
        "tempo_s": tempo_s,
        "parametros": parametros,
    }

### 11.4 Execução da validação aninhada

Em cada fold externo:

1. a Regressão Linear é ajustada no treino externo;
2. o Gradient Boosting executa GridSearchCV somente dentro do treino externo;
3. os melhores parâmetros internos são refitados no treino externo;
4. ambos são avaliados no mesmo fold externo nunca usado no ajuste.

Esse procedimento fornece uma estimativa mais honesta da generalização após fine tuning.

In [ ]:
resultados_validacao = []
previsoes_externas = {
    "Regressão Linear": np.zeros(len(df_atributos)),
    "Gradient Boosting": np.zeros(len(df_atributos)),
}

y_total_real = df_atributos["audience"].to_numpy()
y_total_log = np.log1p(y_total_real)

for fold, (indices_treino, indices_validacao) in enumerate(
    cv_externo.split(df_atributos, estratos), start=1
):
    y_treino_log_fold = y_total_log[indices_treino]
    y_validacao_log_fold = y_total_log[indices_validacao]
    y_treino_real_fold = y_total_real[indices_treino]
    y_validacao_real_fold = y_total_real[indices_validacao]

    # Regressão Linear
    inicio = time.perf_counter()
    modelo_linear_fold = clone(modelo_linear_finalista)
    modelo_linear_fold.fit(
        df_atributos.iloc[indices_treino][atributos_linear_finalista],
        y_treino_log_fold,
    )
    pred_linear_treino = np.maximum(0, np.expm1(modelo_linear_fold.predict(
        df_atributos.iloc[indices_treino][atributos_linear_finalista]
    )))
    pred_linear_validacao = np.maximum(0, np.expm1(modelo_linear_fold.predict(
        df_atributos.iloc[indices_validacao][atributos_linear_finalista]
    )))
    tempo_linear = time.perf_counter() - inicio
    previsoes_externas["Regressão Linear"][indices_validacao] = pred_linear_validacao
    resultados_validacao.append(registrar_metricas_fold(
        "Regressão Linear", fold, y_treino_real_fold, pred_linear_treino,
        y_validacao_real_fold, pred_linear_validacao, tempo_linear, "sem tuning"
    ))

    # Gradient Boosting com busca interna
    inicio = time.perf_counter()
    busca_interna_gb = GridSearchCV(
        GradientBoostingRegressor(random_state=SEED),
        param_grid=grade_gb_aninhada,
        scoring=scorer_mae_real,
        cv=cv_interno,
        n_jobs=-1,
        refit=True,
    )
    busca_interna_gb.fit(
        df_atributos.iloc[indices_treino][atributos_gb_finalista],
        y_treino_log_fold,
    )
    melhor_gb_fold = busca_interna_gb.best_estimator_
    pred_gb_treino = np.maximum(0, np.expm1(melhor_gb_fold.predict(
        df_atributos.iloc[indices_treino][atributos_gb_finalista]
    )))
    pred_gb_validacao = np.maximum(0, np.expm1(melhor_gb_fold.predict(
        df_atributos.iloc[indices_validacao][atributos_gb_finalista]
    )))
    tempo_gb = time.perf_counter() - inicio
    previsoes_externas["Gradient Boosting"][indices_validacao] = pred_gb_validacao
    resultados_validacao.append(registrar_metricas_fold(
        "Gradient Boosting", fold, y_treino_real_fold, pred_gb_treino,
        y_validacao_real_fold, pred_gb_validacao, tempo_gb,
        busca_interna_gb.best_params_,
    ))

tabela_folds = pd.DataFrame(resultados_validacao)
display(tabela_folds.drop(columns="parametros").round(3))

### 11.5 Média, desvio padrão e intervalo do MAE

O intervalo de 95% é calculado com distribuição t e quatro graus de liberdade. Com somente cinco folds, o intervalo tende a ser amplo; ele serve para demonstrar incerteza, não como garantia exata.

Se os intervalos dos modelos se sobrepõem fortemente, uma pequena diferença de média não é evidência suficiente para preferir o modelo mais complexo.

In [ ]:
T_CRITICO_95_4GL = 2.776

resumo_validacao = tabela_folds.groupby("modelo").agg(
    MAE_medio=("MAE", "mean"),
    MAE_dp=("MAE", "std"),
    RMSE_medio=("RMSE", "mean"),
    RMSE_dp=("RMSE", "std"),
    R2_medio=("R2", "mean"),
    R2_dp=("R2", "std"),
    MAPE_medio=("MAPE_%", "mean"),
    MAPE_dp=("MAPE_%", "std"),
    WAPE_medio=("WAPE_%", "mean"),
    WAPE_dp=("WAPE_%", "std"),
    MAE_treino_medio=("MAE_treino", "mean"),
    tempo_medio_s=("tempo_s", "mean"),
).reset_index()

margem = T_CRITICO_95_4GL * resumo_validacao["MAE_dp"] / np.sqrt(5)
resumo_validacao["MAE_IC95_inferior"] = resumo_validacao["MAE_medio"] - margem
resumo_validacao["MAE_IC95_superior"] = resumo_validacao["MAE_medio"] + margem
resumo_validacao["razao_MAE_validacao_treino"] = (
    resumo_validacao["MAE_medio"] / resumo_validacao["MAE_treino_medio"]
)

display(resumo_validacao.round(3))

### 11.6 Métricas agregadas das previsões externas

Além da média dos folds, calculamos métricas sobre todas as previsões externas reunidas. Cada evento foi previsto por um modelo que não o utilizou no treinamento externo.

In [ ]:
metricas_oof_externas = []
for nome_modelo, previsoes in previsoes_externas.items():
    metricas_oof_externas.append({
        "modelo": nome_modelo,
        **calcular_metricas(y_total_real, previsoes),
    })

tabela_oof_externa = pd.DataFrame(metricas_oof_externas).sort_values("MAE")
display(tabela_oof_externa.round(3))

### 11.7 Estabilidade dos hiperparâmetros do Gradient Boosting

Registramos os parâmetros escolhidos em cada fold externo. Se configurações muito diferentes vencerem em cada fold, isso indica sensibilidade da busca à pequena amostra.

In [ ]:
parametros_gb_folds = tabela_folds.loc[
    tabela_folds["modelo"] == "Gradient Boosting",
    ["fold", "parametros"],
]
display(parametros_gb_folds)

### 11.8 Visualização da estabilidade

O gráfico apresenta o MAE dos dois modelos em cada fold. A comparação fold a fold mostra se um modelo vence de maneira consistente ou apenas em algumas divisões.

In [ ]:
PASTA_VALIDACAO = PASTA_PROJETO / "outputs" / "validacao_cruzada"
PASTA_VALIDACAO.mkdir(parents=True, exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=tabela_folds, x="modelo", y="MAE", ax=eixos[0])
sns.stripplot(data=tabela_folds, x="modelo", y="MAE", color="black", size=5, alpha=0.7, ax=eixos[0])
eixos[0].set(title="Distribuição do MAE nos folds externos", xlabel="", ylabel="MAE em pessoas")

resumo_mae_plot = tabela_folds.groupby("modelo", as_index=False)["MAE"].agg(["mean", "std"]).reset_index()
eixos[1].bar(
    resumo_mae_plot["modelo"],
    resumo_mae_plot["mean"],
    yerr=resumo_mae_plot["std"],
    capsize=6,
    color=["#4c78a8", "#f58518"],
)
eixos[1].set(title="MAE médio ± desvio entre folds", xlabel="", ylabel="MAE em pessoas")

fig.tight_layout()
plt.savefig(PASTA_VALIDACAO / "01_mae_por_fold_externo.png", dpi=180, bbox_inches="tight")
plt.show()

### 11.9 Regra de decisão

O modelo final não será escolhido apenas pela menor média. A decisão considera:

1. MAE e RMSE externos;
2. desvio e intervalo de incerteza;
3. diferença treino–validação;
4. estabilidade dos hiperparâmetros;
5. interpretabilidade;
6. tempo de ajuste e inferência;
7. dependência do impacto e viabilidade de produção.

Quando a vantagem do modelo complexo for pequena e incerta, a alternativa simples será preferida. Se o Gradient Boosting mantiver ganho consistente, ele será o modelo acadêmico final, sempre separado do cenário pré-evento.

### 11.10 Exportação

São exportadas métricas por fold, resumo estatístico, previsões externas e parâmetros internos. Esses arquivos permitem auditar a escolha final.

In [ ]:
tabela_folds.to_csv(PASTA_VALIDACAO / "metricas_por_fold.csv", index=False, encoding="utf-8-sig")
resumo_validacao.to_csv(PASTA_VALIDACAO / "resumo_validacao.csv", index=False, encoding="utf-8-sig")
tabela_oof_externa.to_csv(PASTA_VALIDACAO / "metricas_oof_externas.csv", index=False, encoding="utf-8-sig")

previsoes_validacao_exportar = pd.DataFrame({
    "audience_real": y_total_real,
    "porte_avaliacao": df_atributos["porte_avaliacao"].astype(str),
    **previsoes_externas,
})
previsoes_validacao_exportar.to_csv(
    PASTA_VALIDACAO / "previsoes_externas.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Resultados da validação cruzada salvos em: {PASTA_VALIDACAO}")

## 12. Modelo final

A validação aninhada selecionou a Regressão Linear com cinco atributos como modelo acadêmico final. A escolha considera desempenho externo, estabilidade, interpretabilidade e custo.

O modelo final é treinado novamente com todos os 204 eventos para aproveitar a informação disponível. As métricas oficiais permanecem as da validação cruzada; métricas calculadas no próprio treino não seriam estimativas válidas de generalização.

### 12.1 Atributos finais

Os cinco atributos são:

- `log_impacto_nominal`;
- `genero_eletronica`;
- `amplitude_etaria`;
- `genero_outros`;
- `genero_indie`.

O impacto domina a capacidade preditiva e mantém o modelo no cenário acadêmico A. Os indicadores musicais e a amplitude etária representam pequenas correções adicionais.

In [ ]:
atributos_modelo_final = [
    "log_impacto_nominal",
    "genero_eletronica",
    "amplitude_etaria",
    "genero_outros",
    "genero_indie",
]

assert atributos_modelo_final == [
    "log_impacto_nominal",
    "genero_eletronica",
    "amplitude_etaria",
    "genero_outros",
    "genero_indie",
], "A ordem dos atributos finais foi alterada."

print("Atributos finais:")
print(atributos_modelo_final)

### 12.2 Treinamento com toda a base

O pipeline padroniza os atributos e ajusta a Regressão Linear ao alvo `log1p(audience)`. A padronização coloca as variáveis na mesma escala e permite comparar a magnitude dos coeficientes.

In [ ]:
modelo_final = Pipeline([
    ("escala", StandardScaler()),
    ("modelo", LinearRegression()),
])

X_modelo_final = df_atributos[atributos_modelo_final]
y_modelo_final_log = np.log1p(df_atributos["audience"])

inicio_treino_final = time.perf_counter()
modelo_final.fit(X_modelo_final, y_modelo_final_log)
tempo_treino_final_s = time.perf_counter() - inicio_treino_final

print(f"Modelo treinado com {len(X_modelo_final)} eventos.")
print(f"Tempo de treinamento final: {tempo_treino_final_s:.6f} segundo.")

### 12.3 Interpretação dos coeficientes

Como os atributos foram padronizados, cada coeficiente representa a variação esperada em `log1p(audience)` associada ao aumento de um desvio padrão no atributo, mantendo os demais constantes.

Os coeficientes indicam associação, não causalidade. Em especial, o coeficiente do impacto não prova que aumentar impacto econômico causará aumento de público.

In [ ]:
coeficientes_modelo_final = pd.DataFrame({
    "atributo": atributos_modelo_final,
    "coeficiente_padronizado": modelo_final.named_steps["modelo"].coef_,
})
coeficientes_modelo_final["valor_absoluto"] = coeficientes_modelo_final["coeficiente_padronizado"].abs()
coeficientes_modelo_final = coeficientes_modelo_final.sort_values("valor_absoluto", ascending=False)

display(coeficientes_modelo_final.round(4))
print(f"Intercepto na escala log: {modelo_final.named_steps['modelo'].intercept_:.4f}")

### 12.4 Coeficientes padronizados

As barras mostram direção e magnitude na escala padronizada do modelo. Elas ajudam na interpretação, mas não representam importância causal; especialmente, o impacto nominal não deve ser interpretado como uma alavanca controlável.

In [ ]:
PASTA_MODELO_FINAL = PASTA_PROJETO / "outputs" / "modelo_final"
PASTA_MODELO_FINAL.mkdir(parents=True, exist_ok=True)

coeficientes_plot = coeficientes_modelo_final.sort_values("coeficiente_padronizado")
fig, ax = plt.subplots(figsize=(9, 5))
cores_coef = np.where(coeficientes_plot["coeficiente_padronizado"] >= 0, "#2c7fb8", "#d95f0e")
ax.barh(coeficientes_plot["atributo"], coeficientes_plot["coeficiente_padronizado"], color=cores_coef)
ax.axvline(0, color="black", linewidth=1)
ax.set(title="Coeficientes do modelo final", xlabel="Coeficiente padronizado na escala log", ylabel="")
fig.tight_layout()
fig.savefig(PASTA_MODELO_FINAL / "01_coeficientes_modelo_final.png", dpi=180, bbox_inches="tight")
plt.show()

### 12.5 Preparação de uma nova entrada

A função abaixo aplica as mesmas transformações da engenharia de atributos. Ela exige impacto nominal em milhões, faixa etária e gênero.

O valor de impacto ainda não é convertido para uma moeda comum. Portanto, a função serve apenas à demonstração com dados compatíveis com a base atual.

In [ ]:
def preparar_entrada_modelo_final(
    impacto_milhoes_nominais,
    idade_min,
    idade_max,
    genero,
):
    if impacto_milhoes_nominais is None or impacto_milhoes_nominais < 0:
        raise ValueError("impacto_milhoes_nominais deve ser um número não negativo.")
    if idade_min is None or idade_max is None or idade_max < idade_min:
        raise ValueError("A faixa etária informada é inválida.")

    genero_texto = str(genero).strip().lower()
    indicadores = {
        familia: int(any(termo in genero_texto for termo in termos))
        for familia, termos in familias_musicais.items()
    }
    genero_outros = int(sum(indicadores.values()) == 0)

    entrada = pd.DataFrame([{
        "log_impacto_nominal": np.log1p(float(impacto_milhoes_nominais)),
        "genero_eletronica": indicadores["eletronica"],
        "amplitude_etaria": float(idade_max) - float(idade_min),
        "genero_outros": genero_outros,
        "genero_indie": indicadores["indie"],
    }])
    return entrada[atributos_modelo_final]

### 12.5 Função de previsão

O modelo retorna previsão na escala log. A função usa `expm1` para retornar ao número de pessoas e impede resultados negativos.

Uma versão real deveria retornar também intervalo de previsão e alerta de domínio. Nesta demo, a previsão pontual não deve ser interpretada como garantia de comparecimento.

In [ ]:
def prever_publico_modelo_final(
    impacto_milhoes_nominais,
    idade_min,
    idade_max,
    genero,
):
    entrada = preparar_entrada_modelo_final(
        impacto_milhoes_nominais,
        idade_min,
        idade_max,
        genero,
    )
    previsao_log = modelo_final.predict(entrada)[0]
    previsao_real = max(0.0, float(np.expm1(previsao_log)))
    return round(previsao_real)

### 12.6 Teste de funcionamento

O exemplo abaixo é apenas um teste técnico para confirmar que preparação, ordem das colunas e inferência funcionam. Ele não é uma nova avaliação do modelo e não deve ser citado como métrica.

In [ ]:
entrada_teste_tecnico = preparar_entrada_modelo_final(
    impacto_milhoes_nominais=20,
    idade_min=18,
    idade_max=35,
    genero="Electronic/Indie",
)
previsao_teste_tecnico = prever_publico_modelo_final(
    impacto_milhoes_nominais=20,
    idade_min=18,
    idade_max=35,
    genero="Electronic/Indie",
)

display(entrada_teste_tecnico)
print(f"Previsão de demonstração: {previsao_teste_tecnico:,} pessoas".replace(",", "."))

### 12.7 Salvamento do artefato

Salvamos o pipeline com `joblib` e um arquivo de metadados contendo atributos, transformação do alvo, métricas externas, limitações e versão das bibliotecas.

O artefato não inclui a limpeza completa do CSV original. Em produção, o pré-processamento deverá ser transformado em um pipeline versionado que receba campos validados da aplicação.

In [ ]:
import json
import joblib
import sklearn

PASTA_MODELO_FINAL = PASTA_PROJETO / "outputs" / "modelo_final"
PASTA_MODELO_FINAL.mkdir(parents=True, exist_ok=True)

CAMINHO_MODELO_FINAL = PASTA_MODELO_FINAL / "eventinsight_regressao_linear.joblib"
joblib.dump(modelo_final, CAMINHO_MODELO_FINAL)

metricas_lineares_externas = tabela_oof_externa.loc[
    tabela_oof_externa["modelo"] == "Regressão Linear"
].iloc[0]

metadados_modelo = {
    "nome": "EventInsight — Regressão Linear acadêmica",
    "cenario": "A — reprodução acadêmica",
    "quantidade_eventos_treino_final": int(len(df_atributos)),
    "atributos_na_ordem": atributos_modelo_final,
    "transformacao_alvo": "log1p no treino; expm1 na inferência",
    "metricas_validacao_aninhada_oof": {
        "MAE": float(metricas_lineares_externas["MAE"]),
        "RMSE": float(metricas_lineares_externas["RMSE"]),
        "R2": float(metricas_lineares_externas["R2"]),
        "MAPE_pct": float(metricas_lineares_externas["MAPE_%"]),
        "WAPE_pct": float(metricas_lineares_externas["WAPE_%"]),
    },
    "limitacoes": [
        "impacto nominal mistura moedas diferentes",
        "símbolos monetários corrompidos",
        "impacto pode não estar disponível antes do evento",
        "base pequena com 204 eventos",
        "modelo é demonstração acadêmica, não produto em produção",
    ],
    "versoes": {
        "python_sklearn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

(PASTA_MODELO_FINAL / "metadados_modelo.json").write_text(
    json.dumps(metadados_modelo, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

coeficientes_modelo_final.to_csv(
    PASTA_MODELO_FINAL / "coeficientes_modelo.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Modelo salvo em: {CAMINHO_MODELO_FINAL}")

### 12.8 Verificação do arquivo salvo

O modelo é carregado novamente e sua previsão é comparada à previsão anterior. Essa verificação detecta problemas de serialização ou alteração na ordem dos atributos.

In [ ]:
modelo_recarregado = joblib.load(CAMINHO_MODELO_FINAL)
previsao_recarregada = max(
    0.0,
    float(np.expm1(modelo_recarregado.predict(entrada_teste_tecnico)[0])),
)

assert round(previsao_recarregada) == previsao_teste_tecnico
print("Verificação concluída: o modelo salvo reproduziu a previsão de demonstração.")

### 12.9 Limites de uso

O modelo final pode ser usado para:

- demonstrar o pipeline de modelagem;
- reproduzir previsões dentro do contexto da base;
- alimentar as simulações acadêmicas do trabalho;
- comparar evolução metodológica com o Trabalho 1.

Ele não deve ser usado diretamente para:

- aprovar orçamento real;
- definir capacidade de segurança;
- garantir público ou lucro;
- comparar impactos monetários internacionais;
- prever eventos muito diferentes do domínio observado.

Esses limites serão incorporados à análise de viabilidade e ao plano de implantação futura.

## 13. Avaliação por porte

O modelo final é único, mas seu desempenho pode variar conforme a escala do evento. Nesta etapa usamos as previsões externas da validação aninhada, garantindo que cada evento foi previsto sem participar do treinamento externo correspondente.

As três faixas são:

- Pequeno: até 20 mil pessoas;
- Médio: acima de 20 mil e até 80 mil;
- Grande/Mega: acima de 80 mil.

O porte serve somente para avaliação posterior e nunca foi fornecido ao modelo.

### 13.1 Preparação dos erros por evento

Calculamos erro assinado, absoluto e percentual:

- erro assinado = previsto − real;
- valor positivo indica superestimação;
- valor negativo indica subestimação;
- erro absoluto mede magnitude em pessoas;
- erro percentual permite comparar escalas diferentes.

In [ ]:
resultados_por_porte = pd.DataFrame({
    "audience_real": df_atributos["audience"].to_numpy(),
    "audience_previsto": previsoes_externas["Regressão Linear"],
    "porte": df_atributos["porte_avaliacao"],
})

resultados_por_porte["erro_assinado"] = (
    resultados_por_porte["audience_previsto"] - resultados_por_porte["audience_real"]
)
resultados_por_porte["erro_absoluto"] = resultados_por_porte["erro_assinado"].abs()
resultados_por_porte["erro_percentual"] = (
    resultados_por_porte["erro_absoluto"] / resultados_por_porte["audience_real"] * 100
)

display(resultados_por_porte.head())

### 13.2 Métricas por faixa

MAE mostra o erro médio em pessoas. RMSE destaca erros muito grandes. MAPE calcula a média dos percentuais individuais, enquanto WAPE calcula o erro agregado em relação ao público agregado.

Incluímos mediana do erro absoluto porque ela é menos sensível aos maiores eventos. O erro assinado médio identifica direção do viés.

In [ ]:
def metricas_grupo_porte(grupo):
    metricas = calcular_metricas(grupo["audience_real"], grupo["audience_previsto"])
    return pd.Series({
        "N": len(grupo),
        "valores_reais_distintos": grupo["audience_real"].nunique(),
        "MAE": metricas["MAE"],
        "erro_absoluto_mediano": grupo["erro_absoluto"].median(),
        "RMSE": metricas["RMSE"],
        "MAPE_%": metricas["MAPE_%"],
        "WAPE_%": metricas["WAPE_%"],
        "R2": metricas["R2"],
        "erro_assinado_medio": grupo["erro_assinado"].mean(),
    })

tabela_metricas_porte = (
    resultados_por_porte
    .groupby("porte", observed=True)
    .apply(metricas_grupo_porte, include_groups=False)
    .reset_index()
)

tabela_metricas_porte["status_R2"] = np.where(
    tabela_metricas_porte["N"] < 30,
    "Exploratório — amostra pequena",
    "Comparação mais estável",
)

display(tabela_metricas_porte.round(3))

### 13.3 Interpretação do R²

R² compara o erro do modelo com a variabilidade do público dentro de cada faixa. Quando o intervalo do alvo é estreito ou o grupo é pequeno, a métrica pode ser baixa ou negativa mesmo com erros absolutos moderados.

Por isso, o R² de Grande/Mega é mostrado apenas para transparência e recebe o rótulo exploratório. A conclusão desse grupo se apoia principalmente em MAE, RMSE, MAPE, WAPE, erro mediano e `N = 16`.

### 13.4 Comparação entre erro absoluto e percentual

É esperado que o MAE aumente nos eventos maiores, pois existe mais público possível para errar. Entretanto, o MAPE pode diminuir se o erro representar uma parcela menor do evento.

Essa comparação responde à hipótese central do grupo: o mesmo erro absoluto não possui a mesma relevância de negócio em todas as escalas.

In [ ]:
PASTA_AVALIACAO_PORTE = PASTA_PROJETO / "outputs" / "avaliacao_por_porte"
PASTA_AVALIACAO_PORTE.mkdir(parents=True, exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=tabela_metricas_porte, x="porte", y="MAE", ax=eixos[0])
eixos[0].set(title="MAE por faixa", xlabel="Faixa", ylabel="MAE em pessoas")

sns.barplot(data=tabela_metricas_porte, x="porte", y="MAPE_%", ax=eixos[1])
eixos[1].set(title="MAPE por faixa", xlabel="Faixa", ylabel="MAPE (%)")

for eixo in eixos:
    eixo.tick_params(axis="x", rotation=10)

fig.tight_layout()
fig.savefig(PASTA_AVALIACAO_PORTE / "01_mae_mape_por_faixa.png", dpi=180, bbox_inches="tight")
plt.show()

### 13.5 Público real versus previsto por faixa

O gráfico usa escalas logarítmicas para exibir eventos pequenos e grandes no mesmo espaço. A diagonal representa previsão perfeita. A cor mostra a faixa real, utilizada somente após a previsão.

In [ ]:
limite_min_porte = min(
    resultados_por_porte["audience_real"].min(),
    resultados_por_porte["audience_previsto"].min(),
)
limite_max_porte = max(
    resultados_por_porte["audience_real"].max(),
    resultados_por_porte["audience_previsto"].max(),
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=resultados_por_porte,
    x="audience_real",
    y="audience_previsto",
    hue="porte",
    alpha=0.8,
    ax=ax,
)
ax.plot(
    [limite_min_porte, limite_max_porte],
    [limite_min_porte, limite_max_porte],
    "--",
    color="black",
    label="Previsão perfeita",
)
ax.set(
    xscale="log",
    yscale="log",
    title="Público real e previsto por faixa",
    xlabel="Público real",
    ylabel="Público previsto",
)
fig.tight_layout()
fig.savefig(PASTA_AVALIACAO_PORTE / "02_real_previsto_por_faixa.png", dpi=180, bbox_inches="tight")
plt.show()

### 13.6 Viés por faixa

O erro assinado médio permite verificar se o modelo tende a prever acima ou abaixo do público real. Esse diagnóstico é importante para o negócio:

- superestimar público pode levar a custos excessivos e estruturas superdimensionadas;
- subestimar público pode levar a perda de receita, capacidade insuficiente e risco operacional.

O sinal médio deve ser analisado junto à dispersão, pois erros positivos e negativos podem se cancelar.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=tabela_metricas_porte, x="porte", y="erro_assinado_medio", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    title="Erro assinado médio por faixa",
    xlabel="Faixa",
    ylabel="Previsto − real",
)
fig.tight_layout()
fig.savefig(PASTA_AVALIACAO_PORTE / "03_vies_por_faixa.png", dpi=180, bbox_inches="tight")
plt.show()

### 13.7 Limites da conclusão segmentada

A avaliação por faixa permite identificar padrões, mas não transforma os grupos em populações bem representadas. Grande/Mega continua com somente 16 eventos e três ou quatro exemplos por fold externo.

Portanto:

- não treinamos modelos separados;
- não declaramos R² confiável para Grande/Mega;
- não generalizamos os resultados para todos os megaeventos;
- apresentamos quantidade e status junto às métricas;
- tratamos a análise como demonstração da metodologia que será fortalecida com uma base maior.

### 13.8 Exportação

Exportamos a tabela agregada e os erros individuais. A etapa seguinte utilizará os erros individuais para investigar os casos mais difíceis e possíveis padrões de falha.

In [ ]:
tabela_metricas_porte.to_csv(
    PASTA_AVALIACAO_PORTE / "metricas_por_faixa.csv",
    index=False,
    encoding="utf-8-sig",
)
resultados_por_porte.to_csv(
    PASTA_AVALIACAO_PORTE / "erros_individuais_por_faixa.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Avaliação por faixa salva em: {PASTA_AVALIACAO_PORTE}")

## 14. Análise dos erros

A análise de erros investiga padrões que as métricas agregadas não mostram. Utilizamos as previsões externas da Regressão Linear, preservando a avaliação fora da amostra.

Os objetivos são:

- localizar os casos mais difíceis;
- verificar concentração do erro;
- identificar subestimação ou superestimação sistemática;
- investigar relação entre erro e escala;
- diferenciar overfitting global de falhas locais;
- transformar limitações em requisitos para coleta futura.

### 14.1 Tabela detalhada de erros

Reunimos público, previsão, país, gênero, faixa etária, impacto nominal e porte. A tabela mantém contexto suficiente para interpretar cada caso sem usar essas colunas adicionais no modelo.

In [ ]:
analise_erros = resultados_por_porte.copy()
analise_erros["pais"] = df_atributos["pais"].to_numpy()
analise_erros["genero"] = df_atributos["genero"].to_numpy()
analise_erros["faixa_etaria"] = df_atributos["faixa_etaria"].to_numpy()
analise_erros["impacto_milhoes_nominais"] = df_atributos["impacto_milhoes_nominais"].to_numpy()
analise_erros["direcao"] = np.where(
    analise_erros["erro_assinado"] > 0,
    "Superestimação",
    np.where(analise_erros["erro_assinado"] < 0, "Subestimação", "Exato"),
)

display(analise_erros.head())

### 14.2 Maiores erros absolutos

Os maiores erros absolutos mostram quais eventos mais influenciam MAE e RMSE. Eles não são removidos automaticamente: eventos grandes podem ser válidos e possuem alta relevância operacional.

In [ ]:
maiores_erros_absolutos = (
    analise_erros
    .sort_values("erro_absoluto", ascending=False)
    .head(10)
    [[
        "pais", "genero", "porte", "audience_real", "audience_previsto",
        "erro_assinado", "erro_absoluto", "erro_percentual",
        "impacto_milhoes_nominais",
    ]]
)

display(maiores_erros_absolutos.round(2))

### 14.3 Visualização dos dez maiores erros absolutos

O gráfico evidencia a concentração do risco nos casos extremos. A identificação usa país e posição no ranking porque um mesmo país pode aparecer em mais de um evento.

In [ ]:
PASTA_ERROS = PASTA_PROJETO / "outputs" / "analise_erros"
PASTA_ERROS.mkdir(parents=True, exist_ok=True)

top_erros_plot = maiores_erros_absolutos.reset_index(drop=True).copy()
top_erros_plot["evento"] = [f"{pais} ({i + 1})" for i, pais in enumerate(top_erros_plot["pais"])]
top_erros_plot = top_erros_plot.sort_values("erro_absoluto")

fig, ax = plt.subplots(figsize=(10, 6))
cores_top = top_erros_plot["porte"].astype(str).map({
    "Pequeno": "#9ecae1", "Médio": "#4292c6", "Grande/Mega": "#08519c"
})
ax.barh(top_erros_plot["evento"], top_erros_plot["erro_absoluto"], color=cores_top)
ax.set(title="Dez maiores erros absolutos", xlabel="Erro absoluto em pessoas", ylabel="Evento")
fig.tight_layout()
fig.savefig(PASTA_ERROS / "03_top_10_erros_absolutos.png", dpi=180, bbox_inches="tight")
plt.show()

### 14.4 Maiores erros percentuais

O ranking percentual evidencia falhas relevantes em eventos menores que poderiam desaparecer no ranking absoluto. Como não existem públicos iguais a zero, o percentual pode ser calculado diretamente.

In [ ]:
maiores_erros_percentuais = (
    analise_erros
    .sort_values("erro_percentual", ascending=False)
    .head(10)
    [[
        "pais", "genero", "porte", "audience_real", "audience_previsto",
        "erro_assinado", "erro_absoluto", "erro_percentual",
    ]]
)

display(maiores_erros_percentuais.round(2))

### 14.4 Concentração do erro

Calculamos qual parcela da soma dos erros absolutos está concentrada nos cinco e dez casos mais difíceis. Uma concentração alta indica que a métrica global é fortemente influenciada por poucos eventos.

In [ ]:
erros_ordenados = analise_erros["erro_absoluto"].sort_values(ascending=False)
soma_erros = erros_ordenados.sum()

concentracao_erros = pd.DataFrame({
    "grupo": ["5 maiores erros", "10 maiores erros", "Demais eventos"],
    "percentual_do_erro_total": [
        erros_ordenados.head(5).sum() / soma_erros * 100,
        erros_ordenados.head(10).sum() / soma_erros * 100,
        (soma_erros - erros_ordenados.head(10).sum()) / soma_erros * 100,
    ],
})
display(concentracao_erros.round(2))

### 14.5 Subestimação e superestimação

A contagem e a magnitude por direção ajudam a distinguir equilíbrio aparente de impacto real. Muitos erros pequenos positivos podem coexistir com poucos erros negativos muito grandes.

In [ ]:
resumo_direcao = (
    analise_erros
    .groupby("direcao")
    .agg(
        N=("erro_assinado", "size"),
        erro_assinado_medio=("erro_assinado", "mean"),
        erro_absoluto_medio=("erro_absoluto", "mean"),
        erro_percentual_medio=("erro_percentual", "mean"),
    )
    .reset_index()
)
display(resumo_direcao.round(2))

resumo_direcao_porte = (
    analise_erros
    .groupby(["porte", "direcao"], observed=True)
    .size()
    .unstack(fill_value=0)
)
display(resumo_direcao_porte)

### 14.6 Distribuição dos resíduos

Resíduo é definido como previsto menos real. Uma distribuição centrada em zero é desejável, mas assimetria e caudas longas indicam erros sistemáticos ou eventos difíceis.

In [ ]:
PASTA_ERROS = PASTA_PROJETO / "outputs" / "analise_erros"
PASTA_ERROS.mkdir(parents=True, exist_ok=True)

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(analise_erros["erro_assinado"], bins=25, kde=True, ax=eixos[0])
eixos[0].axvline(0, color="black", linewidth=1)
eixos[0].set(title="Distribuição dos resíduos", xlabel="Previsto − real", ylabel="Eventos")

sns.boxplot(data=analise_erros, x="porte", y="erro_assinado", ax=eixos[1])
eixos[1].axhline(0, color="black", linewidth=1)
eixos[1].set(title="Resíduos por faixa", xlabel="Faixa", ylabel="Previsto − real")

fig.tight_layout()
fig.savefig(PASTA_ERROS / "01_distribuicao_residuos.png", dpi=180, bbox_inches="tight")
plt.show()

### 14.7 Resíduos versus previsão

Se a dispersão dos resíduos cresce com a previsão, existe heterocedasticidade: a incerteza não é constante ao longo da escala. Isso indica que um único intervalo de erro não seria adequado para todos os eventos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=analise_erros,
    x="audience_previsto",
    y="erro_assinado",
    hue="porte",
    alpha=0.8,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    xscale="log",
    title="Resíduos versus público previsto",
    xlabel="Público previsto — escala log",
    ylabel="Previsto − real",
)
fig.tight_layout()
fig.savefig(PASTA_ERROS / "02_residuos_vs_previsto.png", dpi=180, bbox_inches="tight")
plt.show()

### 14.8 Relação entre erro e público real

Calculamos correlação de Spearman entre erro absoluto, erro percentual e público real. A correlação é diagnóstica: ela indica se os erros tendem a crescer com a escala, mas não explica a causa.

In [ ]:
correlacao_erros_escala = pd.DataFrame({
    "relação": [
        "Erro absoluto × público real",
        "Erro percentual × público real",
        "Erro assinado × público real",
    ],
    "Spearman": [
        analise_erros["erro_absoluto"].corr(analise_erros["audience_real"], method="spearman"),
        analise_erros["erro_percentual"].corr(analise_erros["audience_real"], method="spearman"),
        analise_erros["erro_assinado"].corr(analise_erros["audience_real"], method="spearman"),
    ],
})
display(correlacao_erros_escala.round(3))

### 14.9 Erro por família musical

As famílias podem se sobrepor: um evento Rock/Pop participa de duas famílias. Calculamos resultados apenas para indicadores com pelo menos dez eventos.

Essa análise é exploratória e não estabelece causalidade. Diferenças podem refletir porte, país ou composição da amostra.

In [ ]:
erros_por_familia = []

for coluna in colunas_genero:
    mascara = df_atributos[coluna] == 1
    if mascara.sum() < 10:
        continue
    erros_por_familia.append({
        "familia": coluna.replace("genero_", ""),
        "N": int(mascara.sum()),
        "MAE": analise_erros.loc[mascara, "erro_absoluto"].mean(),
        "MAPE_%": analise_erros.loc[mascara, "erro_percentual"].mean(),
        "erro_assinado_medio": analise_erros.loc[mascara, "erro_assinado"].mean(),
    })

tabela_erros_familia = pd.DataFrame(erros_por_familia).sort_values("MAE", ascending=False)
display(tabela_erros_familia.round(2))

### 14.10 Overfitting, underfitting e regressão à média

A razão entre MAE externo e de treino da Regressão Linear foi próxima de 1,08, portanto não existe sinal forte de overfitting global.

Entretanto, a forte subestimação de Grande/Mega indica **underfitting local**: uma relação linear dominada pela maioria de eventos pequenos e médios tende a aproximar previsões extremas do centro da distribuição. Esse comportamento é conhecido como regressão à média.

O problema não deve ser corrigido treinando um modelo Grande/Mega com 16 casos. A solução prioritária é ampliar dados e atributos para a maior escala.

### 14.11 Implicações para o produto

A análise recomenda:

- exibir alertas de incerteza para previsões acima de 80 mil;
- não usar previsão pontual como garantia;
- priorizar intervalos de previsão por escala;
- coletar mais eventos Grande/Mega;
- registrar capacidade, vendas antecipadas, atração principal e investimento em mídia;
- monitorar erro assinado por faixa após implantação;
- criar gatilho de revisão humana para eventos fora do domínio comum.

Na demo atual, Grande/Mega deve ser apresentado como faixa não validada para decisão operacional.

### 14.12 Exportação

Os erros detalhados e os resumos são exportados para auditoria e uso no relatório.

In [ ]:
analise_erros.sort_values("erro_absoluto", ascending=False).to_csv(
    PASTA_ERROS / "erros_detalhados.csv",
    index=False,
    encoding="utf-8-sig",
)
maiores_erros_absolutos.to_csv(PASTA_ERROS / "maiores_erros_absolutos.csv", index=False, encoding="utf-8-sig")
maiores_erros_percentuais.to_csv(PASTA_ERROS / "maiores_erros_percentuais.csv", index=False, encoding="utf-8-sig")
concentracao_erros.to_csv(PASTA_ERROS / "concentracao_erros.csv", index=False, encoding="utf-8-sig")
resumo_direcao.to_csv(PASTA_ERROS / "resumo_direcao.csv", index=False, encoding="utf-8-sig")
correlacao_erros_escala.to_csv(PASTA_ERROS / "correlacao_erros_escala.csv", index=False, encoding="utf-8-sig")
tabela_erros_familia.to_csv(PASTA_ERROS / "erros_por_familia.csv", index=False, encoding="utf-8-sig")

print(f"Análise de erros salva em: {PASTA_ERROS}")

## 15. Simulações de negócio

As simulações transformam previsões de público em cenários de planejamento. Entretanto, a base não contém preço real, custos, capacidade ou elasticidade observada.

Por isso:

- impacto econômico não é tratado como custo;
- todos os valores financeiros são expressos em **unidades monetárias hipotéticas (UM)**;
- todas as hipóteses são exibidas;
- os resultados não representam recomendação comercial real;
- o preço encontrado é ótimo somente dentro da função e dos limites definidos.

### 15.1 Público-base da simulação

Para conectar a simulação ao modelo sem usar previsões treinadas sobre o próprio evento, utilizamos a mediana das previsões externas dos eventos Médios.

A mediana reduz a influência de previsões extremas. Esse público-base representa apenas um evento médio hipotético da demo.

In [ ]:
publico_base_simulacao = float(
    resultados_por_porte.loc[
        resultados_por_porte["porte"] == "Médio",
        "audience_previsto",
    ].median()
)

print(f"Público-base da simulação: {publico_base_simulacao:,.0f} pessoas".replace(",", "."))

### 15.2 Função de demanda por preço

Utilizamos uma função de elasticidade constante:

`demanda = público-base × fator do cenário × (preço / preço de referência) ** elasticidade`

A elasticidade é negativa: preços maiores reduzem demanda. A capacidade limita a quantidade atendida.

Essa função é uma hipótese analítica, não uma relação estimada do dataset. Em uma aplicação real, a elasticidade deveria ser estimada com histórico de preços e vendas.

In [ ]:
def estimar_demanda_preco(
    preco,
    publico_base,
    preco_referencia,
    elasticidade,
    fator_demanda,
    capacidade,
):
    demanda = (
        publico_base
        * fator_demanda
        * (preco / preco_referencia) ** elasticidade
    )
    return min(capacidade, max(0.0, demanda))

### 15.3 Cenários e hipóteses

Os três cenários variam simultaneamente demanda, elasticidade e custos. Os valores não são estimativas de mercado; servem para demonstrar análise de sensibilidade.

As elasticidades em valor absoluto acima de 1 tornam a demanda relativamente sensível ao preço e evitam que o ótimo seja definido apenas pelo maior preço da grade.

In [ ]:
PRECO_REFERENCIA = 100.0
CAPACIDADE_HIPOTETICA = 50_000
PRECOS_TESTADOS = np.arange(40, 251, 5)

cenarios_negocio = {
    "Pessimista": {
        "fator_demanda": 0.80,
        "elasticidade": -1.60,
        "custo_fixo": 1_200_000,
        "custo_variavel_pessoa": 35,
    },
    "Base": {
        "fator_demanda": 1.00,
        "elasticidade": -1.30,
        "custo_fixo": 1_000_000,
        "custo_variavel_pessoa": 30,
    },
    "Otimista": {
        "fator_demanda": 1.15,
        "elasticidade": -1.10,
        "custo_fixo": 800_000,
        "custo_variavel_pessoa": 25,
    },
}

display(pd.DataFrame(cenarios_negocio).T)
print(f"Preço de referência: {PRECO_REFERENCIA:.0f} UM")
print(f"Capacidade hipotética: {CAPACIDADE_HIPOTETICA:,} pessoas".replace(",", "."))

### 15.4 Receita, custo e resultado

Para cada preço:

- receita = demanda atendida × preço;
- custo variável = demanda atendida × custo variável por pessoa;
- custo total = custo fixo + custo variável;
- resultado = receita − custo total.

Patrocínio, impostos, taxas de venda, alimentação e outras receitas não foram incluídos. Eles deverão ser adicionados quando existirem dados confiáveis.

In [ ]:
linhas_simulacao = []

for nome_cenario, parametros in cenarios_negocio.items():
    for preco in PRECOS_TESTADOS:
        demanda = estimar_demanda_preco(
            preco=preco,
            publico_base=publico_base_simulacao,
            preco_referencia=PRECO_REFERENCIA,
            elasticidade=parametros["elasticidade"],
            fator_demanda=parametros["fator_demanda"],
            capacidade=CAPACIDADE_HIPOTETICA,
        )
        receita = demanda * preco
        custo_variavel = demanda * parametros["custo_variavel_pessoa"]
        custo_total = parametros["custo_fixo"] + custo_variavel
        resultado = receita - custo_total

        linhas_simulacao.append({
            "cenario": nome_cenario,
            "preco": preco,
            "demanda": demanda,
            "receita": receita,
            "custo_variavel": custo_variavel,
            "custo_fixo": parametros["custo_fixo"],
            "custo_total": custo_total,
            "resultado": resultado,
        })

simulacao_precos = pd.DataFrame(linhas_simulacao)
display(simulacao_precos.head())

### 15.5 Preço ótimo dentro de cada cenário

Selecionamos o preço com maior resultado dentro da grade de 40 a 250 UM. O resultado depende integralmente das hipóteses.

Se a elasticidade, custos, capacidade ou intervalo de preços mudarem, o ótimo também poderá mudar. Portanto, o valor não deve ser apresentado como preço recomendado para um evento real.

In [ ]:
indices_otimos = simulacao_precos.groupby("cenario")["resultado"].idxmax()
precos_otimos_cenarios = (
    simulacao_precos.loc[indices_otimos]
    [["cenario", "preco", "demanda", "receita", "custo_total", "resultado"]]
    .sort_values("cenario")
    .reset_index(drop=True)
)

display(precos_otimos_cenarios.round(2))

### 15.6 Curvas de resultado

As curvas mostram que a otimização não produz somente um número. Elas permitem avaliar regiões de preço com resultados semelhantes e verificar sensibilidade.

Uma região ampla e plana é menos arriscada que um pico estreito, pois pequenos erros na elasticidade produzem menor impacto financeiro.

In [ ]:
PASTA_NEGOCIO = PASTA_PROJETO / "outputs" / "simulacoes_negocio"
PASTA_NEGOCIO.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(
    data=simulacao_precos,
    x="preco",
    y="resultado",
    hue="cenario",
    ax=ax,
)
for _, linha in precos_otimos_cenarios.iterrows():
    ax.scatter(linha["preco"], linha["resultado"], s=60)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    title="Resultado por preço e cenário",
    xlabel="Preço do ingresso (UM)",
    ylabel="Resultado hipotético (UM)",
)
fig.tight_layout()
fig.savefig(PASTA_NEGOCIO / "01_resultado_por_preco.png", dpi=180, bbox_inches="tight")
plt.show()

### 15.7 Ponto de equilíbrio operacional do evento

Para um preço definido, o público de equilíbrio é:

`público de equilíbrio = custo fixo / (preço − custo variável por pessoa)`

O cálculo é válido somente quando o preço supera o custo variável. Ele representa quantidade de ingressos necessária para cobrir os custos considerados no cenário.

In [ ]:
def publico_equilibrio_evento(preco, custo_fixo, custo_variavel_pessoa):
    margem_contribuicao = preco - custo_variavel_pessoa
    if margem_contribuicao <= 0:
        return np.inf
    return custo_fixo / margem_contribuicao

equilibrio_cenarios = []
for _, linha in precos_otimos_cenarios.iterrows():
    parametros = cenarios_negocio[linha["cenario"]]
    equilibrio = publico_equilibrio_evento(
        linha["preco"],
        parametros["custo_fixo"],
        parametros["custo_variavel_pessoa"],
    )
    equilibrio_cenarios.append({
        "cenario": linha["cenario"],
        "preco_otimo_cenario": linha["preco"],
        "publico_equilibrio": equilibrio,
        "demanda_estimada": linha["demanda"],
        "margem_seguranca_pessoas": linha["demanda"] - equilibrio,
    })

tabela_equilibrio_evento = pd.DataFrame(equilibrio_cenarios)
display(tabela_equilibrio_evento.round(2))

### 15.8 Impacto financeiro do erro de público

Para demonstrar como o erro preditivo afeta planejamento, adotamos preço de referência de 100 UM e custo variável de 30 UM. A margem de contribuição hipotética é 70 UM por pessoa.

Erro no resultado planejado = `(público previsto − público real) × margem de contribuição`.

Esse cálculo não estima lucro real; ele traduz o erro de público para uma unidade financeira hipotética.

In [ ]:
MARGEM_CONTRIBUICAO_DEMO = 100 - 30

impacto_erro_financeiro = resultados_por_porte.copy()
impacto_erro_financeiro["erro_resultado_planejado_UM"] = (
    impacto_erro_financeiro["erro_assinado"] * MARGEM_CONTRIBUICAO_DEMO
)

resumo_erro_financeiro_porte = (
    impacto_erro_financeiro
    .groupby("porte", observed=True)
    .agg(
        N=("erro_resultado_planejado_UM", "size"),
        erro_medio_planejamento_UM=("erro_resultado_planejado_UM", "mean"),
        erro_absoluto_medio_planejamento_UM=("erro_resultado_planejado_UM", lambda x: x.abs().mean()),
    )
    .reset_index()
)

display(resumo_erro_financeiro_porte.round(2))

### 15.9 Consequência para Grande/Mega

Como o modelo subestima Grande/Mega em média, a projeção financeira baseada somente na previsão também tende a subestimar receita e margem de contribuição nessa faixa.

Entretanto, o risco mais importante pode ser operacional: contratar estrutura, segurança e capacidade abaixo da demanda real. Por isso, a previsão da demo não deve ser usada isoladamente em eventos de maior escala.

### 15.10 Limitações e dados necessários

Uma otimização real de preço exigiria:

- preços históricos e quantidade vendida;
- vendas por lote e canal;
- capacidade do local;
- taxas, impostos e comissões;
- custos fixos e variáveis reais;
- receitas de patrocínio e consumo;
- cancelamentos e no-show;
- elasticidade por artista, região e público;
- restrições contratuais e regulatórias.

Sem esses dados, a simulação demonstra método e sensibilidade, não recomenda preço.

### 15.11 Exportação

Exportamos hipóteses, grade de preços, ótimos condicionais, ponto de equilíbrio e impacto financeiro dos erros.

In [ ]:
pd.DataFrame(cenarios_negocio).T.to_csv(
    PASTA_NEGOCIO / "hipoteses_cenarios.csv",
    encoding="utf-8-sig",
)
simulacao_precos.to_csv(PASTA_NEGOCIO / "simulacao_grade_precos.csv", index=False, encoding="utf-8-sig")
precos_otimos_cenarios.to_csv(PASTA_NEGOCIO / "otimos_condicionais.csv", index=False, encoding="utf-8-sig")
tabela_equilibrio_evento.to_csv(PASTA_NEGOCIO / "ponto_equilibrio_evento.csv", index=False, encoding="utf-8-sig")
resumo_erro_financeiro_porte.to_csv(
    PASTA_NEGOCIO / "impacto_financeiro_erro_por_porte.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Simulações de negócio salvas em: {PASTA_NEGOCIO}")

## 16. Viabilidade técnica

**Objetivo da etapa:** medir o custo computacional do modelo final e discutir se a solução pode ser implantada e ampliada com segurança.

Esta avaliação combina medições reais no ambiente de execução com análise de arquitetura. Como a base possui somente 204 eventos e cinco atributos finais, os tempos atuais representam uma prova de conceito; não devem ser extrapolados linearmente para uma plataforma completa sem testes de carga.

### 16.1 Critério de medição

Medimos o pipeline final completo, incluindo padronização e regressão. O tempo de treinamento é repetido para reduzir oscilações ocasionais. Na inferência, reutilizamos observações válidas e construímos lotes de 1, 100, 1.000 e 10.000 eventos.

O pico de memória é medido com `tracemalloc`, que acompanha alocações do Python. Ele é útil para comparação local, mas não representa toda a memória residente do processo nem o custo futuro de API, banco de dados e monitoramento.

In [ ]:
from time import perf_counter
import platform
import tracemalloc
from sklearn.base import clone

PASTA_VIABILIDADE_TECNICA = PASTA_PROJETO / "outputs" / "viabilidade_tecnica"
PASTA_VIABILIDADE_TECNICA.mkdir(parents=True, exist_ok=True)

REPETICOES_TREINO = 30
tempos_treino_s = []

for _ in range(REPETICOES_TREINO):
    modelo_benchmark = clone(modelo_final)
    inicio = perf_counter()
    modelo_benchmark.fit(X_modelo_final, y_modelo_final_log)
    tempos_treino_s.append(perf_counter() - inicio)

tracemalloc.start()
modelo_memoria = clone(modelo_final)
modelo_memoria.fit(X_modelo_final, y_modelo_final_log)
memoria_atual_bytes, memoria_pico_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()

tamanho_modelo_bytes = CAMINHO_MODELO_FINAL.stat().st_size

resumo_treinamento = pd.DataFrame([{
    "eventos_treino": len(X_modelo_final),
    "atributos": len(atributos_modelo_final),
    "repeticoes": REPETICOES_TREINO,
    "tempo_medio_ms": np.mean(tempos_treino_s) * 1000,
    "tempo_mediano_ms": np.median(tempos_treino_s) * 1000,
    "tempo_dp_ms": np.std(tempos_treino_s) * 1000,
    "memoria_pico_tracemalloc_kb": memoria_pico_bytes / 1024,
    "arquivo_modelo_kb": tamanho_modelo_bytes / 1024,
}])

display(resumo_treinamento.round(4))

### 16.2 Escalabilidade da inferência

O modelo é carregado apenas uma vez e recebe lotes em memória. Para cada tamanho, registramos a mediana de várias execuções e calculamos o tempo por evento e a vazão. Usar a mediana reduz a influência do aquecimento e de processos concorrentes.

In [ ]:
tamanhos_lote = [1, 100, 1_000, 10_000]
REPETICOES_INFERENCIA = 30
resultados_inferencia = []

# Aquecimento para evitar que a primeira chamada distorça o resultado.
_ = modelo_recarregado.predict(X_modelo_final.iloc[:2])

for tamanho in tamanhos_lote:
    lote = X_modelo_final.sample(
        n=tamanho,
        replace=tamanho > len(X_modelo_final),
        random_state=SEED,
    ).reset_index(drop=True)

    tempos = []
    for _ in range(REPETICOES_INFERENCIA):
        inicio = perf_counter()
        previsoes_log = modelo_recarregado.predict(lote)
        _ = np.expm1(previsoes_log)
        tempos.append(perf_counter() - inicio)

    tempo_mediano_s = float(np.median(tempos))
    resultados_inferencia.append({
        "tamanho_lote": tamanho,
        "repeticoes": REPETICOES_INFERENCIA,
        "tempo_mediano_ms": tempo_mediano_s * 1000,
        "tempo_por_evento_ms": tempo_mediano_s * 1000 / tamanho,
        "eventos_por_segundo": tamanho / tempo_mediano_s,
    })

benchmark_inferencia = pd.DataFrame(resultados_inferencia)
display(benchmark_inferencia.round(6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.lineplot(
    data=benchmark_inferencia,
    x="tamanho_lote",
    y="tempo_mediano_ms",
    marker="o",
    ax=axes[0],
)
axes[0].set_xscale("log")
axes[0].set_title("Tempo total por tamanho do lote")
axes[0].set_xlabel("Eventos no lote (escala log)")
axes[0].set_ylabel("Tempo mediano (ms)")

sns.lineplot(
    data=benchmark_inferencia,
    x="tamanho_lote",
    y="tempo_por_evento_ms",
    marker="o",
    ax=axes[1],
)
axes[1].set_xscale("log")
axes[1].set_title("Custo médio por evento")
axes[1].set_xlabel("Eventos no lote (escala log)")
axes[1].set_ylabel("ms por evento")

plt.tight_layout()
plt.show()

### 16.3 Interpretação da complexidade

Com $n$ eventos e $p$ atributos, a regressão linear possui custo de ajuste aproximadamente associado à solução de mínimos quadrados, dependente de $n$ e $p$, enquanto cada previsão cresce linearmente com $p$. Aqui, $p=5$, portanto treinamento e inferência são leves.

Além da velocidade, a regressão escolhida é menor e mais interpretável que modelos de árvores em conjunto. Isso reforça a decisão tomada na validação aninhada: o modelo final não foi escolhido somente pelo menor erro, mas pelo equilíbrio entre generalização, estabilidade, explicabilidade e custo operacional.

### 16.4 Arquitetura proposta para produção

1. Uma interface recebe os atributos conhecidos do evento.
2. Uma API valida tipos, intervalos e campos obrigatórios.
3. O mesmo pré-processamento salvo no pipeline transforma os dados.
4. O modelo retorna público previsto, versão do modelo e avisos de confiança.
5. Previsões e resultados reais posteriores são registrados para monitoramento.
6. Eventos previstos ou caracterizados como Grande/Mega são enviados para revisão humana.

Para a demonstração, uma aplicação e uma API podem funcionar em uma única instância de CPU, sem GPU. Em crescimento, API, banco de dados e monitoramento podem ser separados e escalados horizontalmente. O modelo deve permanecer carregado em memória, evitando reler o arquivo a cada requisição.

### 16.5 Monitoramento e retreinamento

O sistema deve acompanhar:

- erros MAE, WAPE e erro assinado globalmente e por porte;
- proporção de previsões fora dos intervalos observados no treino;
- mudanças na distribuição dos atributos e do público;
- latência, falhas da API e volume de requisições;
- versão dos dados, do código e do modelo;
- desempenho específico de Grande/Mega.

O retreinamento não deve ocorrer apenas por calendário. Ele deve ser considerado quando houver volume suficiente de novos eventos representativos, degradação confirmada em dados já realizados ou mudança relevante do domínio. Todo candidato deve repetir a validação temporal ou externa antes de substituir o modelo ativo.

### 16.6 Limitações técnicas e decisão

A implementação é computacionalmente viável como prova de conceito, mas ainda não é operacionalmente confiável para decisões autônomas. Os principais bloqueios são a base pequena, a baixa representação de Grande/Mega, a ausência de variáveis pré-evento importantes e a dependência do cenário principal em impacto econômico nominal.

Assim, a recomendação é uma implantação inicial assistida: previsões para apoio analítico, registro obrigatório do resultado real e revisão humana em situações de maior risco. A ampliação da infraestrutura só se justifica depois da ampliação e melhoria dos dados.

In [ ]:
ambiente_execucao = pd.DataFrame([{
    "sistema": platform.system(),
    "versao_sistema": platform.version(),
    "processador": platform.processor() or "não informado pelo ambiente",
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
}])

resumo_treinamento.to_csv(
    PASTA_VIABILIDADE_TECNICA / "benchmark_treinamento.csv", index=False, encoding="utf-8-sig"
)
benchmark_inferencia.to_csv(
    PASTA_VIABILIDADE_TECNICA / "benchmark_inferencia.csv", index=False, encoding="utf-8-sig"
)
ambiente_execucao.to_csv(
    PASTA_VIABILIDADE_TECNICA / "ambiente_execucao.csv", index=False, encoding="utf-8-sig"
)

display(ambiente_execucao)
print(f"Resultados técnicos salvos em: {PASTA_VIABILIDADE_TECNICA}")

## 17. Viabilidade econômica

**Objetivo da etapa:** estimar custos, receitas, ponto de equilíbrio, retorno e sustentabilidade econômica do produto EventInsight.

Esta seção avalia o **sistema de previsão como produto SaaS**, e não o lucro de um festival. A etapa 15 simulou decisões dentro de um evento; aqui analisamos desenvolvimento, hospedagem, dados, manutenção, aquisição de clientes e assinaturas.

Como não há dados contábeis nem pesquisa de mercado na base, todos os valores estão em unidade monetária hipotética (UM). Os resultados são análise de cenários e não previsão financeira auditada.

### 17.1 Modelo de receita e horizonte

Adotamos assinatura mensal com receita média por cliente (ARPU). A projeção cobre **60 meses**, permitindo observar maturação, ponto de equilíbrio e sensibilidade a aquisição e cancelamento.

O número de clientes evolui por:

\[
Clientes_t = Clientes_{t-1}(1-Churn) + NovosClientes_t
\]

Receita, custo e caixa são calculados por:

\[
Receita_t = Clientes_t \times ARPU
\]

\[
ResultadoOperacional_t = Receita_t - CustosFixos_t - Marketing_t - Clientes_t \times CustoVariavel
\]

O investimento inicial inclui construção da aplicação, integração, testes e preparação da primeira versão. Ele é descontado no primeiro mês do fluxo de caixa.

In [ ]:
PASTA_VIABILIDADE_ECONOMICA = PASTA_PROJETO / "outputs" / "viabilidade_economica"
PASTA_VIABILIDADE_ECONOMICA.mkdir(parents=True, exist_ok=True)

HORIZONTE_MESES = 60

cenarios_produto = {
    "Pessimista": {
        "clientes_iniciais": 5,
        "novos_clientes_mes": 2,
        "churn_mensal": 0.040,
        "arpu_mensal_UM": 250,
        "investimento_inicial_UM": 120_000,
        "custo_fixo_mensal_UM": 12_000,
        "custo_variavel_cliente_UM": 40,
        "marketing_mensal_UM": 3_000,
    },
    "Base": {
        "clientes_iniciais": 10,
        "novos_clientes_mes": 5,
        "churn_mensal": 0.025,
        "arpu_mensal_UM": 350,
        "investimento_inicial_UM": 150_000,
        "custo_fixo_mensal_UM": 18_000,
        "custo_variavel_cliente_UM": 45,
        "marketing_mensal_UM": 5_000,
    },
    "Otimista": {
        "clientes_iniciais": 15,
        "novos_clientes_mes": 10,
        "churn_mensal": 0.015,
        "arpu_mensal_UM": 450,
        "investimento_inicial_UM": 180_000,
        "custo_fixo_mensal_UM": 25_000,
        "custo_variavel_cliente_UM": 55,
        "marketing_mensal_UM": 8_000,
    },
}

hipoteses_economicas = pd.DataFrame(cenarios_produto).T.reset_index(names="cenario")
display(hipoteses_economicas)

### 17.2 Justificativa das hipóteses

Os cenários variam simultaneamente preço médio, aquisição, churn e estrutura de custos. Isso evita concluir viabilidade a partir de uma única combinação favorável.

- **Pessimista:** adesão lenta, maior cancelamento e ticket menor.
- **Base:** crescimento moderado e equipe enxuta para operação contínua.
- **Otimista:** maior capacidade comercial e ticket superior, acompanhados por mais investimento, equipe e marketing.

O custo variável representa processamento, armazenamento, suporte e serviços proporcionais ao cliente. O custo fixo agrega equipe, administração, manutenção, ferramentas e infraestrutura mínima. Valores reais exigiriam cotações, salários locais, impostos e validação de disposição a pagar.

In [ ]:
composicao_custo_fixo_base = pd.DataFrame({
    "categoria": [
        "Equipe técnica e manutenção",
        "Infraestrutura e monitoramento",
        "Aquisição/licenciamento de dados",
        "Ferramentas e licenças",
        "Administrativo e contingência",
    ],
    "custo_mensal_UM": [11_000, 1_500, 2_000, 1_000, 2_500],
})

assert composicao_custo_fixo_base["custo_mensal_UM"].sum() == cenarios_produto["Base"]["custo_fixo_mensal_UM"]
display(composicao_custo_fixo_base)

### 17.3 Composição do custo fixo base

Usamos barras, e não pizza, porque elas permitem comparar valores próximos com mais precisão. O gráfico descreve hipóteses do cenário base, não despesas observadas.

In [ ]:
custos_base_plot = composicao_custo_fixo_base.sort_values("custo_mensal_UM")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(custos_base_plot["categoria"], custos_base_plot["custo_mensal_UM"], color="#4c78a8")
ax.set(title="Composição hipotética do custo fixo mensal — cenário base", xlabel="Custo mensal (UM)", ylabel="")
fig.tight_layout()
plt.show()

### 17.4 Projeção mensal de clientes e caixa

Não arredondamos clientes durante a recorrência para preservar a expectativa matemática do cenário; nas apresentações, os valores podem ser exibidos arredondados. Também não incluímos impostos, inflação, desconto financeiro ou financiamento, pois não existem informações suficientes para estimá-los de forma defensável.

In [ ]:
linhas_projecao = []

for nome_cenario, h in cenarios_produto.items():
    clientes = float(h["clientes_iniciais"])
    caixa_acumulado = -float(h["investimento_inicial_UM"])

    for mes in range(1, HORIZONTE_MESES + 1):
        if mes > 1:
            clientes = clientes * (1 - h["churn_mensal"]) + h["novos_clientes_mes"]

        receita = clientes * h["arpu_mensal_UM"]
        custo_variavel = clientes * h["custo_variavel_cliente_UM"]
        custo_operacional = h["custo_fixo_mensal_UM"] + h["marketing_mensal_UM"] + custo_variavel
        resultado_operacional = receita - custo_operacional
        caixa_acumulado += resultado_operacional

        linhas_projecao.append({
            "cenario": nome_cenario,
            "mes": mes,
            "ano": int(np.ceil(mes / 12)),
            "clientes": clientes,
            "receita_UM": receita,
            "custo_variavel_UM": custo_variavel,
            "custo_operacional_UM": custo_operacional,
            "resultado_operacional_UM": resultado_operacional,
            "caixa_acumulado_UM": caixa_acumulado,
        })

projecao_mensal = pd.DataFrame(linhas_projecao)
display(projecao_mensal.head())

### 17.4 Break-even, ROI e payback

Usamos dois conceitos diferentes:

- **break-even operacional:** quantidade de clientes necessária para que a receita mensal cubra custos fixos, marketing e custo variável;
- **payback:** primeiro mês em que o caixa acumulado recupera o investimento inicial.

O ROI de 60 meses é calculado como resultado operacional acumulado menos investimento, dividido pelo investimento. Quando não há recuperação dentro do horizonte, o payback é informado como “não atingido em 60 meses”.

In [ ]:
resumos_economicos = []

for nome_cenario, h in cenarios_produto.items():
    dados = projecao_mensal.query("cenario == @nome_cenario").copy()
    margem_contribuicao_cliente = h["arpu_mensal_UM"] - h["custo_variavel_cliente_UM"]
    clientes_break_even = (
        h["custo_fixo_mensal_UM"] + h["marketing_mensal_UM"]
    ) / margem_contribuicao_cliente

    meses_caixa_positivo = dados.loc[dados["caixa_acumulado_UM"] >= 0, "mes"]
    payback_mes = int(meses_caixa_positivo.iloc[0]) if not meses_caixa_positivo.empty else np.nan

    resultado_operacional_60m = dados["resultado_operacional_UM"].sum()
    resultado_liquido_60m = resultado_operacional_60m - h["investimento_inicial_UM"]
    roi_60m = resultado_liquido_60m / h["investimento_inicial_UM"] * 100

    resumos_economicos.append({
        "cenario": nome_cenario,
        "clientes_break_even": clientes_break_even,
        "mes_break_even_operacional": (
            int(dados.loc[dados["clientes"] >= clientes_break_even, "mes"].iloc[0])
            if (dados["clientes"] >= clientes_break_even).any()
            else np.nan
        ),
        "payback_mes": payback_mes,
        "receita_60m_UM": dados["receita_UM"].sum(),
        "custos_operacionais_60m_UM": dados["custo_operacional_UM"].sum(),
        "investimento_inicial_UM": h["investimento_inicial_UM"],
        "resultado_liquido_60m_UM": resultado_liquido_60m,
        "ROI_60m_percentual": roi_60m,
        "clientes_mes_60": dados.iloc[-1]["clientes"],
    })

resumo_viabilidade_economica = pd.DataFrame(resumos_economicos)
display(resumo_viabilidade_economica.round(2))

### 17.5 Visão anual e gráficos

A agregação anual evita avaliar sustentabilidade apenas por um mês isolado. O caixa acumulado mostra se e quando o investimento inicial é recuperado, enquanto clientes e resultado operacional mostram a trajetória que produziu esse retorno.

In [ ]:
projecao_anual = (
    projecao_mensal
    .groupby(["cenario", "ano"], as_index=False)
    .agg(
        clientes_fim_ano=("clientes", "last"),
        receita_anual_UM=("receita_UM", "sum"),
        custo_operacional_anual_UM=("custo_operacional_UM", "sum"),
        resultado_operacional_anual_UM=("resultado_operacional_UM", "sum"),
        caixa_acumulado_fim_ano_UM=("caixa_acumulado_UM", "last"),
    )
)
display(projecao_anual.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

sns.lineplot(data=projecao_mensal, x="mes", y="clientes", hue="cenario", ax=axes[0])
axes[0].set_title("Evolução dos clientes")
axes[0].set_ylabel("Clientes esperados")

sns.lineplot(data=projecao_mensal, x="mes", y="resultado_operacional_UM", hue="cenario", ax=axes[1], legend=False)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Resultado operacional mensal")
axes[1].set_ylabel("Resultado (UM)")

sns.lineplot(data=projecao_mensal, x="mes", y="caixa_acumulado_UM", hue="cenario", ax=axes[2], legend=False)
axes[2].axhline(0, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Caixa acumulado após investimento")
axes[2].set_ylabel("Caixa acumulado (UM)")

plt.tight_layout()
plt.show()

### 17.6 Sustentabilidade e análise crítica

Um cenário lucrativo não prova que o negócio será viável: aquisição constante de clientes, churn baixo e disposição a pagar precisam ser validados. Além disso, o modelo atual depende de uma base pequena e seu cenário de melhor desempenho usa impacto econômico nominal, variável que pode não estar disponível antes do evento.

Antes de investimento comercial relevante, são necessários:

- entrevistas e piloto pago com organizadores;
- validação do preço e dos planos;
- orçamento real de equipe, nuvem, impostos, suporte e dados;
- métricas de custo de aquisição (CAC), valor do cliente (LTV) e churn observado;
- base maior com atributos genuinamente disponíveis antes do evento;
- limites de responsabilidade e comunicação clara da incerteza.

A decisão recomendada é um piloto controlado. A expansão deve ocorrer somente se qualidade preditiva e indicadores comerciais forem confirmados conjuntamente.

### 17.7 Exportação

Exportamos hipóteses e projeções para permitir auditoria e substituição posterior por valores reais. Isso é mais transparente que deixar números financeiros fixos apenas no texto.

In [ ]:
hipoteses_economicas.to_csv(
    PASTA_VIABILIDADE_ECONOMICA / "hipoteses_cenarios_produto.csv", index=False, encoding="utf-8-sig"
)
composicao_custo_fixo_base.to_csv(
    PASTA_VIABILIDADE_ECONOMICA / "composicao_custo_fixo_base.csv", index=False, encoding="utf-8-sig"
)
projecao_mensal.to_csv(
    PASTA_VIABILIDADE_ECONOMICA / "projecao_mensal_60_meses.csv", index=False, encoding="utf-8-sig"
)
projecao_anual.to_csv(
    PASTA_VIABILIDADE_ECONOMICA / "projecao_anual.csv", index=False, encoding="utf-8-sig"
)
resumo_viabilidade_economica.to_csv(
    PASTA_VIABILIDADE_ECONOMICA / "resumo_break_even_roi_payback.csv", index=False, encoding="utf-8-sig"
)

print(f"Resultados econômicos salvos em: {PASTA_VIABILIDADE_ECONOMICA}")

## 18. Conclusões

**Objetivo da etapa:** justificar o modelo final e sintetizar evolução técnica, limitações, viabilidade e próximos passos.

Esta seção não realiza novo treinamento nem escolhe o modelo olhando novamente para os mesmos resultados. Ela consolida as evidências das etapas anteriores para evitar uma conclusão baseada em uma única métrica.

### 18.1 Resumo automático das evidências

A tabela abaixo recupera os resultados já produzidos. Essa rastreabilidade reduz o risco de o texto final apresentar valores diferentes daqueles efetivamente calculados no notebook.

In [ ]:
linha_validacao_final = resumo_validacao.query("modelo == 'Regressão Linear'").iloc[0]
linha_tecnica_final = resumo_treinamento.iloc[0]
linha_economica_base = resumo_viabilidade_economica.query("cenario == 'Base'").iloc[0]

resumo_executivo = pd.DataFrame([
    {"dimensao": "Dados", "indicador": "Eventos após limpeza", "valor": len(df_atributos), "unidade": "eventos"},
    {"dimensao": "Modelo", "indicador": "Atributos finais", "valor": len(atributos_modelo_final), "unidade": "atributos"},
    {"dimensao": "Validação externa", "indicador": "MAE médio", "valor": linha_validacao_final["MAE_medio"], "unidade": "pessoas"},
    {"dimensao": "Validação externa", "indicador": "RMSE médio", "valor": linha_validacao_final["RMSE_medio"], "unidade": "pessoas"},
    {"dimensao": "Validação externa", "indicador": "R² médio", "valor": linha_validacao_final["R2_medio"], "unidade": "proporção"},
    {"dimensao": "Validação externa", "indicador": "MAPE médio", "valor": linha_validacao_final["MAPE_medio"], "unidade": "%"},
    {"dimensao": "Validação externa", "indicador": "WAPE médio", "valor": linha_validacao_final["WAPE_medio"], "unidade": "%"},
    {"dimensao": "Generalização", "indicador": "Razão MAE validação/treino", "valor": linha_validacao_final["razao_MAE_validacao_treino"], "unidade": "razão"},
    {"dimensao": "Operação", "indicador": "Treinamento mediano", "valor": linha_tecnica_final["tempo_mediano_ms"], "unidade": "ms"},
    {"dimensao": "Operação", "indicador": "Tamanho do modelo", "valor": linha_tecnica_final["arquivo_modelo_kb"], "unidade": "KB"},
    {"dimensao": "Economia — cenário base", "indicador": "Payback", "valor": linha_economica_base["payback_mes"], "unidade": "mês"},
    {"dimensao": "Economia — cenário base", "indicador": "ROI em 60 meses", "valor": linha_economica_base["ROI_60m_percentual"], "unidade": "% hipotético"},
])

display(resumo_executivo.round(3))

### 18.2 Modelo selecionado e justificativa

O modelo acadêmico final é uma **Regressão Linear aplicada a `log1p(audience)`**, com padronização e cinco atributos:

- log do impacto econômico nominal;
- indicador de música eletrônica;
- amplitude da faixa etária;
- indicador de outros gêneros;
- indicador de indie.

Na validação cruzada aninhada, a Regressão Linear apresentou MAE médio próximo de 9,3 mil pessoas, RMSE de 22,6 mil, R² de 0,798, MAPE de 15,81% e WAPE de 21,90%. A razão entre MAE de validação e treino ficou próxima de 1,08, sem evidência forte de overfitting global.

O Gradient Boosting permaneceu como candidato relevante, mas sua vantagem preliminar não se manteve na avaliação externa: teve erro médio superior e maior diferença entre treino e validação. A Regressão Linear foi escolhida pelo melhor equilíbrio entre generalização, estabilidade, interpretação, tempo de inferência e simplicidade de implantação.

### 18.3 Evolução do Trabalho 1 para o Trabalho 2

O Trabalho 2 não apenas adicionou algoritmos. Ele tornou o processo mais controlado e auditável.

In [ ]:
evolucao_trabalhos = pd.DataFrame([
    ["Carregamento", "Leitura direta e interpretação inicial", "Validação estrutural e correção do deslocamento semântico das colunas"],
    ["Limpeza", "Tratamento básico", "Duplicata removida, tipos corrigidos e moedas preservadas sem conversão artificial"],
    ["Atributos", "Dependência principal do impacto", "Engenharia de faixa etária, log do impacto e famílias de gênero"],
    ["Feature selection", "Não sistematizada", "Correlação, importâncias, permutação, LassoCV e RFECV somente no treino"],
    ["Modelos", "Comparação inicial", "Quatro famílias, conjuntos de atributos e dois cenários de disponibilidade"],
    ["Ajuste", "Configurações limitadas", "GridSearchCV e validação aninhada sem contaminar a avaliação externa"],
    ["Porte", "Hipótese de modelos separados", "Modelo único e avaliação por Pequeno, Médio e Grande/Mega"],
    ["Erros", "Métricas globais", "Viés, concentração dos erros, extremos e avaliação segmentada"],
    ["Negócio", "Simulação mais direta", "Hipóteses explícitas, cenários, break-even, ROI, payback e limitações"],
    ["Implantação", "Proposta conceitual", "Benchmark, arquitetura, monitoramento, revisão humana e retreinamento"],
], columns=["aspecto", "trabalho_1", "trabalho_2"])

display(evolucao_trabalhos)

### 18.4 O que os resultados permitem concluir

O modelo demonstra que existe sinal preditivo na base e produz uma melhoria clara sobre uma referência ingênua. A transformação logarítmica ajudou a tratar a forte assimetria do público e a validação aninhada forneceu uma estimativa mais honesta de generalização.

A avaliação por porte, entretanto, mostrou que desempenho global não significa segurança uniforme:

- **Pequeno:** MAE de aproximadamente 1.853 pessoas e MAPE de 12,64%;
- **Médio:** MAE de aproximadamente 6.937 pessoas e MAPE de 15,39%;
- **Grande/Mega:** MAE de aproximadamente 63.453 pessoas e MAPE de 34,84%, com somente 16 eventos.

Em Grande/Mega, 11 de 16 eventos foram subestimados. Portanto, a hipótese inicial de que erros absolutos maiores seriam percentualmente pequenos não foi confirmada. O grupo deve permanecer unido apenas para avaliação exploratória; a amostra não sustenta modelos separados por porte.

### 18.5 Limitações determinantes

As principais limitações são:

1. base reduzida, com 204 eventos após limpeza;
2. somente 16 observações em Grande/Mega;
3. alta cardinalidade e muitas categorias raras de país e gênero;
4. impacto econômico registrado em moedas diferentes, com símbolos corrompidos e sem data confiável;
5. forte dependência do impacto nominal no cenário de melhor desempenho;
6. poucos atributos realmente disponíveis antes da realização do evento;
7. ausência de capacidade, preço, vendas antecipadas, artista, marketing, clima e histórico;
8. dados sem organização temporal adequada para simular produção real;
9. hipóteses financeiras sem validação contábil ou de mercado.

O cenário B, sem impacto econômico, obteve desempenho muito inferior. Isso revela que o modelo final é adequado como demonstração acadêmica, mas ainda não resolve de forma robusta a previsão antecipada exigida por um produto real.

### 18.6 Viabilidade técnica, econômica e operacional

**Técnica:** o pipeline é extremamente leve, treina em poucos milissegundos, ocupa cerca de 1,54 KB e não requer GPU. A infraestrutura não é o principal obstáculo; os dados são.

**Econômica:** o cenário base hipotético atingiu break-even operacional no mês 18 e payback no mês 46, enquanto o pessimista permaneceu inviável em 60 meses. Isso demonstra potencial condicionado, não rentabilidade comprovada. Preço, aquisição, churn, impostos, salários e disposição a pagar ainda precisam ser observados.

**Operacional:** o sistema não deve automatizar decisões de orçamento, segurança ou capacidade. A implantação inicial deve ser assistida, com revisão humana obrigatória para Grande/Mega e comunicação de incerteza ao usuário.

### 18.7 Recomendação de implantação

A decisão recomendada é prosseguir apenas como **prova de conceito e piloto controlado**:

1. padronizar um formulário de coleta pré-evento;
2. ampliar prioritariamente eventos acima de 80 mil pessoas;
3. registrar previsões, decisões e públicos reais sem sobrescrever versões;
4. apresentar intervalo de previsão e alerta de domínio, além do valor pontual;
5. exigir revisão humana em Grande/Mega;
6. validar utilidade com organizadores e iniciar piloto pago limitado;
7. acompanhar erro por porte, drift, latência, churn, CAC e LTV;
8. retreinar somente após volume representativo e nova validação externa ou temporal.

Uma arquitetura simples de API em CPU é suficiente para o piloto. Escalar infraestrutura antes de validar dados e mercado aumentaria custos sem resolver a principal fonte de risco.

### 18.8 Conclusão final

O Trabalho 2 demonstrou evolução técnica clara em relação ao Trabalho 1: seleção de atributos mais rigorosa, comparação sistemática de modelos, ajuste sem vazamento, validação aninhada, análise por porte, diagnóstico dos erros e avaliação de viabilidade.

A Regressão Linear foi a escolha mais defensável para a base atual. Ela combina desempenho, estabilidade, interpretação e baixo custo. Contudo, o projeto ainda não está pronto para decisões reais autônomas. A contribuição principal é uma prova de conceito reproduzível que identifica tanto o potencial do modelo quanto as condições necessárias para avançar.

O próximo investimento deve ser feito na qualidade, diversidade e disponibilidade temporal dos dados. Com uma base maior e atributos pré-evento relevantes, o mesmo processo de validação poderá determinar se a solução evolui de demonstração acadêmica para produto confiável e economicamente sustentável.

In [ ]:
PASTA_CONCLUSOES = PASTA_PROJETO / "outputs" / "conclusoes"
PASTA_CONCLUSOES.mkdir(parents=True, exist_ok=True)

resumo_executivo.to_csv(PASTA_CONCLUSOES / "resumo_executivo.csv", index=False, encoding="utf-8-sig")
evolucao_trabalhos.to_csv(PASTA_CONCLUSOES / "evolucao_trabalho_1_para_2.csv", index=False, encoding="utf-8-sig")

print("Conclusão consolidada sem novo ajuste do modelo.")
print(f"Tabelas finais salvas em: {PASTA_CONCLUSOES}")